# Reproducing Danish, Aronov & Fee (2017) from CRCNS aa-3

**Paper.** Danish HH, Aronov D, Fee MS (2017). Rhythmic syllable-related activity in a songbird
motor thalamic nucleus necessary for learned vocalizations. *PLoS ONE* 12(6): e0169568.
[doi:10.1371/journal.pone.0169568](https://doi.org/10.1371/journal.pone.0169568). The claims
file (`claims/danish-2017.yaml`) was written from the version of record (PMC5472270), with the
legends of the supporting information.

**Data set.** [CRCNS aa-3](https://doi.org/10.6080/K0GQ6VP4), read from the public bucket
`crcnsarchive` under `aa-3/live/`: `Electrophysiology.tar.gz` (4,254,324,000 bytes, MD5
0a93b3d4c7c2f98e8d29ffcb91842db4) and `Lesion_Experiments.tar.gz` (5,134,663,426 bytes, MD5
0d4f55eba659b1e3976c3220663e6e9e), as listed in the data set's
`checksums.md5`. CRCNS data sets are not versioned; the version recorded is `unversioned`, and
these checksums pin what was read. The paper cites the data set without a version, and there is
only this one copy (`aa-3/live/`; there is no `aa-3/old/`). The two archives are streamed once
each (twice for the lesion archive) and reduced as they are read; nothing is written to disk.

**What the data set holds.** Multiunit recordings in Uva during singing from 3 birds (the paper
recorded 5), with the authors' segmentations and labels, and one day of song before and one day
after bilateral Uva lesions from 7 birds (the paper's 7 Uva-lesioned birds). There are no
control-lesion birds, no directed-song labels in the lesion data, and no sorted single units.

| Claim | Tested here |
|---|---|
| lesion-directed-song-abolished | blocked: metadata insufficient (no recording is marked as directed song) |
| lesion-undirected-rate-similar | tested |
| fig2g-maturity-index-drop | tested |
| fig2f-closer-to-exponential | tested |
| fig2c-small-syllable-peak | tested |
| fig2d-gaps-broader | tested |
| fig2e-rhythmicity-drop | tested |
| fig2e-residual-rhythm-peak | tested |
| figs1-control-lesions-no-effect | blocked: data missing (no control-lesion birds) |
| fig3-antidromic | blocked: other (no stimulus times and no identified units; the artefacts are described below) |
| single-unit-spontaneous-rates | blocked: metadata insufficient (no sorted or identified units) |
| fig4c-call-premotor | tested |
| fig4b-introductory-note-premotor | tested |
| fig4a-elevated-during-song | tested |
| fig5-consistent-across-renditions | tested |
| fig5-consistent-across-sites | tested |
| fig6-peak-before-onset | tested |
| fig6-dip-before-offset | tested |
| fig7a-correlation-with-amplitude | tested |
| spectral-features-weak | tested; the attempt record keeps the verified outcome, blocked, and proposes not reproduced |
| fig7b-10hz-power-peaks | tested |
| fig7d-10hz-coherence | tested |
| fig8-no-gte-relation | blocked: other (GTE extraction not attempted) |
| fig9-long-syllable-gamma | tested |
| bout-offset-suppression | tested |
| no-motif-bout-signal | tested |

In [ ]:
import io
import time
import resource
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import uva

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()
eph = uva.load_ephys()
print(f"Electrophysiology archive streamed and reduced in {time.time() - started:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## Data check: electrophysiology

In [ ]:
files = uva.file_table(eph)
sheet = uva.sheet_table(eph["sheets"])
print("Archive members read:", len(eph["md5"]))
print("Analysis files (dbase) per bird and day, with the recordings each lists:")
print(files.groupby(["bird", "day"]).agg(files=("file", "size"), loaded=("loaded", "sum"),
      minutes=("seconds", lambda s: round(s.sum() / 60, 1)), segments=("n_seg", "sum"),
      selected=("n_selected", "sum"), song_syllables=("n_song_syl", "sum"), intro=("n_intro", "sum"),
      calls=("n_call", "sum"), outside_uva=("outside", "sum"), with_stim=("stim_artefacts", "sum")).to_string())
fs_all = sorted({round(r["fs"], 3) for r in eph["recs"].values()})
print("\nSampling rates of the .dat files (Hz):", fs_all, "; data description: 40,000")
print("Files whose closing sample count is unreadable (rate taken as 40 kHz):", eph.get("bad_tail", []))
print("Neural channel numbers per bird:", {k[0]: r["chans"] for k, r in eph["recs"].items()})
nest = eph["nested_md5"]
same = sum(eph["md5"].get("Electrophysiology/Bird_2-Orange150/" + k) == v for k, v in nest.items())
print(f"\nNested archive Bird_2-Orange150/2013-10-06.tar.gz: {len(nest)} files, {same} identical (MD5) "
      "to the files of the folder Bird_2-Orange150/2013-10-06; it is a second copy and is not used.")
dat_md5 = pd.Series({k: v for k, v in eph["md5"].items() if k.endswith(".dat")})
print("Distinct .dat files with identical content elsewhere in the archive:", int(dat_md5.duplicated(keep=False).sum()))
print("\nSpreadsheet rows per sheet tab:")
print(sheet.groupby(["bird", "tab"]).agg(rows=("file", "size"), first=("file", "min"), last=("file", "max"),
      depths=("depth", "nunique")).to_string())
print("\nSpreadsheet notes (top 15):")
print(sheet.note.value_counts().head(15).to_string())

In [ ]:
lab = files.groupby(["bird"]).agg(n=("file", "size")).index.tolist()
titles = {}
for (bird, day), db in eph["dbase"].items():
    for f in db["files"]:
        for t, s in zip(f["titles"], f["selected"]):
            titles.setdefault(bird, {}).setdefault((t if t else "(none)", bool(s)), 0)
            titles[bird][(t if t else "(none)", bool(s))] += 1
for bird, c in titles.items():
    print(bird, "segment labels (label, selected): count")
    print("   ", dict(sorted(c.items(), key=lambda kv: -kv[1])))
# Durations of the labels, to say what they are.
rows = []
for bird in sorted(titles):
    for ev in uva.events(eph, bird):
        for t, a, b, s in zip(ev["title"], ev["on"], ev["off"], ev["sel"]):
            rows.append({"bird": bird, "label": t if t else "(none)", "selected": s, "dur": b - a})
lab = pd.DataFrame(rows)
print("\nMedian duration (ms) by bird and label, selected segments:")
print(lab[lab.selected].groupby(["bird", "label"]).dur.agg(["size", "median"]).round(0).to_string())

The labels are not described in the data descriptor beyond "syllable name". The data say what
they are: letters a to h (Orange150), a to d (b871) and a to f (Uva4) are the motif syllables,
"i" occurs at the start of bouts before "a" (introductory notes), and "k" is a sound of about
210 ms that makes up the files the spreadsheets mark as "distance calls" (b871, Uva4). The
descriptor says that introductory notes and calls are unselected; in the files, "i" and "k" are
selected. Orange150 has no call labels.

**Which neural channel.** Each recording has a sound channel and three neural channels (5, 6 and
7, or 1, 2 and 3 for Uva4). The paper describes a single electrode, and neither the descriptor
nor the files say which channel is the Uva electrode (the plotting script asks the user to
choose 1 to 3). The cell below shows, for each bird and channel, the syllable-onset-aligned MUA.
The channel used is, for each bird, the one with the largest range of the onset-aligned average
(an assumption; stated with the claims that rest on it, which show the other channels too). The
analysis files list files for all three channels (`ChannelFiles`, which the data set's
`process_recording.m` reads as the neural channels), so they do not settle it. One analysis
file, Orange150's first day, ran its event detectors (threshold crossings and stimulus
detection) on channel 7, the channel chosen here for that bird. The choice uses the syllable
modulation that later claims measure; the fig6 section shows the latencies on every channel.

In [ ]:
def song_events(bird, exclude_outside=True):
    """Song files of a bird: selected syllables other than introductory notes and calls."""
    out = []
    for ev in uva.events(eph, bird):
        if exclude_outside and any(ev["props"].get(p, False) for p in uva.OUTSIDE_FLAGS):
            continue
        song = ev["sel"] & ~np.isin(ev["title"], ["i", "k", ""])
        if song.sum() >= 3:
            out.append(ev)
    return out


BIRDS = ["Bird_1-b871", "Bird_2-Orange150", "Bird_3-Uva4"]
chan_rows = []
for bird in BIRDS:
    evs = song_events(bird)
    nch = len(eph["recs"][evs[0]["key"]]["chans"])
    for ci in range(nch):
        rows_ = []
        for ev in evs:
            r = eph["recs"][ev["key"]]
            m = r["mua"][ci]
            song = ev["sel"] & ~np.isin(ev["title"], ["i", "k", ""])
            a, _ = uva.aligned(m / m.mean(), ev["on"][song], 150, 150)
            rows_.append(a)
        A = np.concatenate(rows_).mean(0)
        chan_rows.append({"bird": bird, "channel": eph["recs"][evs[0]["key"]]["chans"][ci],
                          "onsets": sum(len(x) for x in rows_), "range": A.max() - A.min(),
                          "peak_ms": int(np.argmax(A[100:150]) - 50), "mean_-150_-100": A[:50].mean()})
chan_tab = pd.DataFrame(chan_rows)
print("Syllable-onset-aligned MUA (each file's MUA divided by its mean), per channel:")
print(chan_tab.round(3).to_string(index=False))
CHAN = {b: int(chan_tab[chan_tab.bird == b].sort_values("range").iloc[-1].channel) for b in BIRDS}
CI = {b: eph["recs"][song_events(b)[0]["key"]]["chans"].index(CHAN[b]) for b in BIRDS}
print("\nChannel used per bird:", CHAN)
print("\nWhat the analysis files say about the neural channels (ChannelFiles lists that are not empty, which "
      "process_recording.m of the data set reads as the neural channels; event detectors and their source):")
for (bird, day), db in sorted(eph["dbase"].items()):
    print(f"  {bird} {day}: channels with files {sorted(db['channel_files'])}; "
          f"event detectors {db['event_detectors']} on {db['event_sources']}")
r0 = eph["recs"][song_events("Bird_2-Orange150")[0]["key"]]
cc = np.corrcoef(np.stack([r0["mua"][i] for i in range(3)]))
print("Correlation of the three MUA traces in one Orange150 song file:\n", cc.round(3))

**Clock check.** Sound and neural channels are separate files per recording, each with its own
header. They share a clock if the syllable labels, made on the sound, line up with the neural
signal. The cell below decodes song from MUA: within song files, the MUA in 50 ms windows
before syllable onsets against the MUA at random times inside bouts, with the true onsets and
with the onsets shifted by 0.5 to 2 s.

In [ ]:
rng = np.random.default_rng(11)
for bird in BIRDS:
    out = {}
    for shift in [0, 500, 1000, 2000]:
        auc = []
        for ev in song_events(bird):
            m = eph["recs"][ev["key"]]["mua"][CI[bird]]
            song = ev["sel"] & ~np.isin(ev["title"], ["i", "k", ""])
            on = ev["on"][song] + shift
            on = on[(on > 60) & (on < len(m) - 1)]
            if len(on) < 3:
                continue
            pre = np.array([m[int(t) - 40:int(t)].mean() for t in on])
            inb = np.concatenate([np.arange(int(a), int(b)) for a, b, _ in ev["bouts"]])
            inb = inb[(inb > 60) & (inb < len(m))]
            rnd = np.array([m[t - 40:t].mean() for t in rng.choice(inb, len(on))])
            auc.append(np.mean(pre[:, None] > rnd[None, :]))
        out[shift] = np.mean(auc)
    print(bird, "probability that pre-onset MUA exceeds MUA at a random time in a bout; by onset shift (ms):",
          {k: round(v, 3) for k, v in out.items()})

## Electrophysiology claims

Common choices for this part, each from the claims file or the paper:
- MUA is the chosen channel of each bird (assumption above), from song files at sites inside Uva
  (files whose properties mark them above, below or outside Uva are left out).
- Bouts are vocal segments with gaps of at most 300 ms (Methods, Sound analysis).
- The shuffled baseline is reading (a) of the claims file header: averages of the same number of
  windows taken at random times inside bouts (for the bout-offset claim, outside singing), 1000
  times; the band is the 2.5 and 97.5 percentiles of the shuffles' minima and maxima over the
  window (-150 to +150 ms here, a standard choice). Reading (b), MUA circularly shifted within
  each bout, is run for the syllable-onset and offset claims.
- Values are per bird and then averaged over the 3 birds; the paper averages over 5.

In [ ]:
PRE, POST = 150, 150
LAGS = np.arange(-PRE, POST + 1)


def bird_traces(bird, ci=None):
    ci = CI[bird] if ci is None else ci
    return [(ev, eph["recs"][ev["key"]]["mua"][ci]) for ev in song_events(bird)]


def inside_bout_pool(bird, ci=None):
    pool = []
    for ev, m in bird_traces(bird, ci):
        t = np.concatenate([np.arange(int(a), int(b)) for a, b, _ in ev["bouts"]])
        t = t[(t >= PRE) & (t < len(m) - POST - 1)]
        if len(t):
            pool.append((m, t))
    return pool


def event_average(bird, times_of, ci=None):
    rows = []
    for ev, m in bird_traces(bird, ci):
        a, _ = uva.aligned(m, times_of(ev), PRE, POST)
        rows.append(a)
    A = np.concatenate(rows)
    return A


def circ_shift_band(bird, times_of, n_shuffle, rng, ci=None):
    """Shuffled baseline reading (b): MUA circularly shifted within each bout relative to the
    event times; band of the shuffles' minima and maxima (2.5 and 97.5 percentiles). The windows
    are cut from the shifted bout taken as periodic, so the 150 ms before and after each event
    come from shifted bout activity too (events at a bout's start see the bout's shifted end)."""
    items = []
    for ev, m in bird_traces(bird, ci):
        t = times_of(ev)
        for a, b, _ in ev["bouts"]:
            a, b = int(a), int(b)
            sel = t[(t >= a) & (t < b)]
            if len(sel) and a - PRE >= 0 and b + POST < len(m):
                items.append((m[a - PRE:b + POST], sel - a + PRE, b - a))
    mins, maxs, means = [], [], []
    for _ in range(n_shuffle):
        acc = np.zeros(PRE + POST + 1); n = 0
        for seg, tt, L in items:
            core = seg[PRE:PRE + L]
            sh = np.roll(core, rng.integers(L))
            full = np.take(sh, np.arange(-PRE, L + POST), mode="wrap")
            a, ok = uva.aligned(full, tt, PRE, POST)
            acc += a.sum(0); n += len(a)
        avg = acc / n
        means.append(avg.mean()); mins.append(avg.min()); maxs.append(avg.max())
    return float(np.mean(means)), np.percentile(mins, 2.5), np.percentile(maxs, 97.5)


AVG = {}  # (event name, bird, shuffle) -> (average, level, band low, band high), for the figure


def latency_table(times_of, direction, n_shuffle=1000, seed=0, readings=("a",), ci_of=None, name=None):
    rows = []
    for bird in BIRDS:
        rng = np.random.default_rng(seed)
        ci = None if ci_of is None else ci_of[bird]
        A = event_average(bird, times_of, ci)
        avg = A.mean(0)
        n_ev = len(A)
        for rd in readings:
            if rd == "a":
                level, lo, hi = uva.shuffle_band(inside_bout_pool(bird, ci), n_ev, PRE, POST, n_shuffle, rng)
            else:
                level, lo, hi = circ_shift_band(bird, times_of, n_shuffle, rng, ci)
            if name:
                AVG[(name, bird, rd)] = (avg, level, lo, hi)
            thr = hi if direction == "up" else lo
            t_ext, t_cross = uva.crossing_before(avg, LAGS, 0, thr, direction)
            rows.append({"bird": bird, "shuffle": rd, "events": n_ev, "extreme_ms": t_ext,
                         "leaves_band_ms": t_cross, "avg_level": level, "band_lo": lo, "band_hi": hi,
                         "extreme_value": avg[LAGS == t_ext][0]})
    return pd.DataFrame(rows)

### fig4c-call-premotor

> Uva activity rises sharply just before distance calls, leaving baseline 30 ms and peaking 15 ms
> before call onset.

Calls are the segments labelled "k" (b871 and Uva4; Orange150 has none). Calls occur outside
song bouts, so the baseline windows are drawn from the call files themselves (random times in
the call files) as well as from inside bouts, the claims file's reading; both are shown.

In [ ]:
def call_events(bird):
    out = []
    for ev in uva.events(eph, bird):
        if any(ev["props"].get(p, False) for p in uva.OUTSIDE_FLAGS):
            continue
        k = ev["sel"] & (ev["title"] == "k")
        if k.sum():
            out.append((ev, eph["recs"][ev["key"]]["mua"][CI[bird]], ev["on"][k]))
    return out


rows = []
for bird in BIRDS:
    ce = call_events(bird)
    if not ce:
        rows.append({"bird": bird, "calls": 0}); continue
    A = np.concatenate([uva.aligned(m, t, PRE, POST)[0] for _, m, t in ce])
    avg = A.mean(0)
    for base in ["inside bouts (claims file)", "call files"]:
        rng = np.random.default_rng(1)
        if base.startswith("inside"):
            level, lo, hi = uva.shuffle_band(inside_bout_pool(bird), len(A), PRE, POST, 1000, rng)
        else:
            pool = [(m, np.arange(PRE, len(m) - POST - 1)) for _, m, _ in ce]
            level, lo, hi = uva.shuffle_band(pool, len(A), PRE, POST, 1000, rng)
        t_pk, t_cross = uva.crossing_before(avg, LAGS, 0, hi, "up")
        if base.startswith("inside"):
            AVG[("calls", bird, "a")] = (avg, level, lo, hi)
        rows.append({"bird": bird, "calls": len(A), "baseline": base, "peak_ms": t_pk, "leaves_band_ms": t_cross,
                     "peak": avg[LAGS == t_pk][0], "band_hi": hi, "avg_level": level})
calls_tab = pd.DataFrame(rows)
print(calls_tab.round(4).to_string(index=False))
for base in calls_tab.baseline.dropna().unique():
    s = calls_tab[calls_tab.baseline == base]
    crossed = s[s.leaves_band_ms.notna()].bird.tolist()
    print(f"{base}: latency of peak, mean over {s.peak_ms.notna().sum()} birds with calls, {-s.peak_ms.mean():.1f} ms "
          f"(paper 15 +- 5); latency from baseline, mean over the {len(crossed)} birds that leave the band "
          f"{crossed}: {-s.leaves_band_ms.mean():.1f} ms (paper 30 +- 7)")

**Result: partially reproduced (assumed channel).** The call-aligned MUA peaks 13 ms (b871, 145
calls) and 17 ms (Uva4, 86 calls) before onset, mean 15.0 ms over the two birds with labelled
calls (paper 15 +- 5). Only Uva4 leaves the inside-bout band before onset, at 22 ms (paper 30 +-
7), so the latency from baseline is that one bird's value; b871 never exceeds the band (peak
0.0486 against 0.0712), and neither bird does against the band drawn from the call files.
Orange150 has no calls labelled.


### fig4b-introductory-note-premotor

> Uva is robustly active before each introductory note, leaving baseline 39 ms and peaking 21 ms
> before note onset.

In [ ]:
def intro_on(ev):
    return ev["on"][ev["sel"] & (ev["title"] == "i")]


intro_tab = latency_table(intro_on, "up", readings=("a", "b"), name="introductory notes")
print(intro_tab.round(4).to_string(index=False))
for rd in ["a", "b"]:
    s = intro_tab[intro_tab.shuffle == rd]
    crossed = s[s.leaves_band_ms.notna()].bird.tolist()
    print(f"shuffle {rd}: peak, mean over 3 birds, {-s.extreme_ms.mean():.1f} ms (paper 21 +- 9); latency from "
          f"baseline, mean over the {len(crossed)} birds that leave the band {crossed}: "
          f"{-s.leaves_band_ms.mean():.1f} ms (paper 39 +- 8)")

**Result: partially reproduced (assumed channel).** The introductory-note-aligned MUA peaks 23.3 ms
before onset, averaged over the 3 birds (paper 21 +- 9). Under shuffle reading (b), with the
150 ms around each note now cut from the shifted bout, Orange150 (113 notes) and Uva4 (10) leave
the band 36 and 38 ms before onset (mean 37.0 ms over these two birds; paper 39 +- 8); b871 (20
notes) does not leave it. Under reading (a) no bird does, as the band for maxima is wide with so
few notes.


### fig4a-elevated-during-song

> During bouts of singing Uva activity is persistently elevated and modulated in a way locked to
> song.

Per recording site (depth), mean MUA inside bouts against mean MUA more than 300 ms from any
vocal segment in the same files. Sites are the depths of the spreadsheets (b871, Orange150) or
the depth properties of the analysis file (Uva4).

In [ ]:
def site_of(ev):
    bird, day, f = ev["key"]
    if bird == "Bird_3-Uva4":
        d = [k for k, v in ev["props"].items() if v and k.replace(".", "").isdigit()]
        return d[0] if d else None
    s = sheet[(sheet.bird == bird) & (sheet.day == day) & (sheet.file == f)]
    return f"{day} {s.depth.iloc[0]}" if len(s) and np.isfinite(s.depth.iloc[0]) else None


rows = []
for bird in BIRDS:
    for ci in range(3):
        for ev, m in bird_traces(bird, ci):
            alls = np.zeros(len(m), bool)
            for a, b in zip(ev["on"], ev["off"]):
                alls[max(0, int(a) - 300):int(b) + 300] = True
            inb = np.zeros(len(m), bool)
            for a, b, _ in ev["bouts"]:
                inb[int(a):int(b)] = True
            if (~alls).sum() < 200 or inb.sum() < 200:
                continue
            rows.append({"bird": bird, "channel": eph["recs"][ev["key"]]["chans"][ci], "site": site_of(ev),
                         "inside": m[inb].mean(), "outside": m[~alls].mean(), "ms_out": int((~alls).sum())})
el = pd.DataFrame(rows)
site_el = el.groupby(["bird", "channel", "site"]).agg(files=("inside", "size"), inside=("inside", "mean"),
                                                     outside=("outside", "mean")).reset_index()
site_el["ratio"] = site_el.inside / site_el.outside
main_el = site_el[[CHAN[b] == c for b, c in zip(site_el.bird, site_el.channel)]]
print("Chosen channels, per site:")
print(main_el.round(4).to_string(index=False))
t = stats.ttest_rel(main_el.inside, main_el.outside)
print(f"\nSites higher inside bouts: {(main_el.inside > main_el.outside).sum()} of {len(main_el)}; "
      f"paired t-test over sites t = {t.statistic:.2f}, p = {t.pvalue:.2g}")
print("\nAll channels, sites higher inside bouts / sites:")
print(site_el.groupby(["bird", "channel"]).apply(lambda d: f"{(d.inside > d.outside).sum()}/{len(d)}, median ratio {d.ratio.median():.2f}").to_string())

**Result: reproduced (assumed channel).** On the chosen channels, mean MUA inside bouts is higher
than outside singing at all 18 sites (paired t-test over sites p = 1.8e-06; ratios 1.04 to 2.13).
The other channels are higher at most sites too, by less.

### fig6-peak-before-onset and fig6-dip-before-offset

> Uva activity rises significantly before syllable onsets in every bird, peaking 20 ms before
> onset and rising above its average level during singing 32 ms before onset.

> Uva activity falls significantly before syllable offsets in every bird, reaching a minimum
> 18 ms before offset and dropping below its average level during singing 48 ms before offset.

Syllables are the selected song syllables (motif letters; introductory notes and calls are
separate claims).

In [ ]:
def syl_mask(ev):
    return ev["sel"] & ~np.isin(ev["title"], ["i", "k", ""])


def syl_on(ev):
    return ev["on"][syl_mask(ev)]


def syl_off(ev):
    return ev["off"][syl_mask(ev)]


on_tab = latency_table(syl_on, "up", readings=("a", "b"), name="syllable onsets")
print(on_tab.round(4).to_string(index=False))
for rd in ["a", "b"]:
    s = on_tab[on_tab.shuffle == rd]
    print(f"shuffle {rd}: mean over birds, peak {-s.extreme_ms.mean():.1f} ms (SEM {s.extreme_ms.sem():.1f}), "
          f"rises above band {-s.leaves_band_ms.mean():.1f} ms (SEM {s.leaves_band_ms.sem():.1f}) "
          "(paper 20 +- 2 and 32 +- 5, SEM)")


def paired_tests(win_of, ref, bird):
    """Per rendition: mean MUA in the window against the rendition's bout mean (a) or the
    reference window (b)."""
    x, ya, yb = [], [], []
    for ev, m in bird_traces(bird):
        idx = np.where(syl_mask(ev))[0]
        for i in idx:
            w0, w1 = win_of(ev, i)
            if w0 < 0 or w1 > len(m):
                continue
            bt = [(a, b) for a, b, _ in ev["bouts"] if a <= ev["on"][i] <= b]
            r = ref(ev, i)
            if not bt or r is None or r[0] < 0 or r[1] > len(m):
                continue
            x.append(m[int(w0):int(w1)].mean())
            ya.append(m[int(bt[0][0]):int(bt[0][1])].mean())
            yb.append(m[int(r[0]):int(r[1])].mean())
    x, ya, yb = map(np.array, (x, ya, yb))
    ta, tb = stats.ttest_rel(x, ya), stats.ttest_rel(x, yb)
    return {"bird": bird, "n": len(x), "mean_window": x.mean(), "mean_bout": ya.mean(), "t_a": ta.statistic,
            "p_a": ta.pvalue, "mean_ref_b": yb.mean(), "t_b": tb.statistic, "p_b": tb.pvalue}


def prev_offset_window(ev, i):
    if i == 0:
        return None
    o = ev["off"][i - 1]
    return (o - 20, o + 20)


# The same latencies on every channel (shuffle a), to show how they depend on the channel choice.
ch_rows = []
for ci in range(3):
    t_ = latency_table(syl_on, "up", readings=("a",), ci_of={b: ci for b in BIRDS})
    for _, r in t_.iterrows():
        ch_rows.append({"bird": r.bird, "channel": eph["recs"][song_events(r.bird)[0]["key"]]["chans"][ci],
                        "chosen": ci == CI[r.bird], "peak_ms": r.extreme_ms, "rises_above_band_ms": r.leaves_band_ms})
ch_lat = pd.DataFrame(ch_rows)
print("\nOnset latencies on each channel (shuffle a):")
print(ch_lat.to_string(index=False))
print("Channels whose average rises above the band before onset:",
      ch_lat.dropna(subset=["rises_above_band_ms"])[["bird", "channel"]].values.tolist())

tests_on = pd.DataFrame([paired_tests(lambda ev, i: (ev["on"][i] - 40, ev["on"][i]), prev_offset_window, b)
                         for b in BIRDS])
print("\nPaired t-tests, 40 ms before onset against (a) the bout mean, (b) 40 ms centred on the preceding offset:")
print(tests_on.to_string(index=False))

**Result: partially reproduced (assumed channel).** MUA before syllable onsets is higher than the
bout mean and than the preceding offset in every bird (paired t-tests, p below 1e-24 in each
bird, both readings). Averaged over the 3 birds the peak is 17.0 ms before onset (paper 20 +- 2;
outside the tolerance) and the average rises above the band 28.0 ms (shuffle a) or 29.0 ms
(shuffle b) before onset (paper 32 +- 5; within). On the other channels no average rises above
the band before onset; b871's channel 7 peaks 18 ms before onset against 13 ms on channel 6, so
the peak time depends on the channel.


In [ ]:
off_tab = latency_table(syl_off, "down", readings=("a", "b"), name="syllable offsets")
print(off_tab.round(4).to_string(index=False))
for rd in ["a", "b"]:
    s = off_tab[off_tab.shuffle == rd]
    print(f"shuffle {rd}: mean over birds, minimum {-s.extreme_ms.mean():.1f} ms (SEM {s.extreme_ms.sem():.1f}), "
          f"drops below band {-s.leaves_band_ms.mean():.1f} ms (SEM {s.leaves_band_ms.sem():.1f}) "
          "(paper 18 +- 1 and 48 +- 8, SEM)")


def next_onset_window(ev, i):
    if i + 1 >= len(ev["on"]):
        return None
    o = ev["on"][i + 1]
    return (o - 40, o)


tests_off = pd.DataFrame([paired_tests(lambda ev, i: (ev["off"][i] - 50, ev["off"][i]), next_onset_window, b)
                          for b in BIRDS])
print("\nPaired t-tests, 50 ms before offset against (a) the bout mean, (b) the 40 ms before the next onset:")
print(tests_off.to_string(index=False))

**Result: reproduced (assumed channel).** MUA in the 50 ms before syllable offsets is lower than
the bout mean and than the 40 ms before the next onset in every bird (p below 1e-7 in each). The
minimum averages 18.0 ms before offset (paper 18 +- 1) and the average drops below the band 40.7
ms before offset under shuffle reading (b) (paper 48 +- 8; 36.3 ms under reading a, outside).


### fig5-consistent-across-renditions and fig5-consistent-across-sites

> Uva activity is consistent across renditions of the song motif; time-warped MUA is correlated
> across renditions (0.64 with 5 ms smoothing) and coherent below 12 Hz (mean coherence 0.68
> between 1 and 12 Hz).

> Uva activity is similar at different recording depths along a penetration; correlation 0.62
> with 5 ms smoothing and mean coherence 0.65 between 1 and 12 Hz, with no evidence that activity
> varies across sites.

The motif of each bird is its most common sequence of motif letters (printed). A rendition is a
run of consecutive segments spelling the motif, each selected. MUA is warped onto the template
(the rendition whose length is closest to the median; Methods, Time warping), with the
syllable onsets and offsets as alignment points. For the correlation the 2 ms MUA is smoothed
further to a 5 ms total SD. Coherence uses one Slepian taper with time half-bandwidth 2 on the
2 ms MUA; the null is 1000 random circular shifts of each rendition (Rhythmicity).

In [ ]:
from collections import Counter


def motif_of(bird):
    c = Counter()
    for ev in song_events(bird):
        s = "".join(t if (sel and t not in ("i", "k", "")) else "|" for t, sel in zip(ev["title"], ev["sel"]))
        for part in s.split("|"):
            letters = sorted(set(part))
            n = len(letters)
            for j in range(len(part) - n + 1):
                c[part[j:j + n]] += 1
    full = max((m for m in c if len(set(m)) == len(m)), key=lambda m: (len(m) >= 4, c[m] * len(m)))
    return full, c[full]


renditions = {}
for bird in BIRDS:
    motif, n = motif_of(bird)
    rows = []
    for ev, m in bird_traces(bird):
        tt = list(ev["title"])
        for j in range(len(tt) - len(motif) + 1):
            if "".join(tt[j:j + len(motif)]) == motif and ev["sel"][j:j + len(motif)].all():
                pts = np.ravel(np.c_[ev["on"][j:j + len(motif)], ev["off"][j:j + len(motif)]])
                if np.all(np.diff(pts) > 0) and pts[0] >= 0 and pts[-1] < len(m):
                    rows.append((ev, m, pts))
    lens = np.array([p[-1] - p[0] for _, _, p in rows])
    tmpl = rows[int(np.argmin(np.abs(lens - np.median(lens))))][2]
    renditions[bird] = {"motif": motif, "rows": rows, "template": tmpl - tmpl[0], "lens": lens}
    print(f"{bird}: motif '{motif}', {len(rows)} renditions, length median {np.median(lens):.0f} ms "
          f"(range {lens.min():.0f} to {lens.max():.0f})")


def warped(bird, smooth_total_sd=None):
    R = renditions[bird]
    out = []
    for ev, m, pts in R["rows"]:
        x = m if smooth_total_sd is None else uva.gauss_smooth(m, np.sqrt(smooth_total_sd ** 2 - 2 ** 2))
        out.append(uva.warp(x, pts, R["template"]))
    return np.stack(out)


def corr_readings(W):
    Z = (W - W.mean(1, keepdims=True)) / W.std(1, keepdims=True)
    C = Z @ Z.T / W.shape[1]
    iu = np.triu_indices(len(W), 1)
    a = C[iu].mean()
    # (b) peak of the cross-correlation within +-50 ms, averaged over pairs (up to 300 pairs)
    rng = np.random.default_rng(5)
    pairs = np.array(iu).T
    pairs = pairs[rng.choice(len(pairs), min(300, len(pairs)), replace=False)]
    L = W.shape[1]
    b = np.mean([max(np.corrcoef(Z[i][max(0, s):L + min(0, s)], Z[j][max(0, -s):L - max(0, s)])[0, 1]
                     for s in range(-50, 51, 2)) for i, j in pairs])
    c = np.mean([np.corrcoef(W[i], np.delete(W, i, 0).mean(0))[0, 1] for i in range(len(W))])
    return a, b, c


def coherence_readings(W, n_null=1000, seed=0):
    nfft = 2 ** int(np.ceil(np.log2(W.shape[1])) + 2)
    f = np.fft.rfftfreq(nfft, 1 / 1000)
    X = uva.mt_fft(W, nfft)
    band = (f >= 1) & (f <= 12)
    n = len(W)
    iu = np.triu_indices(n, 1)
    if len(iu[0]) > 2000:
        sel = np.random.default_rng(seed).choice(len(iu[0]), 2000, replace=False)
        iu = (iu[0][sel], iu[1][sel])
    Ca = np.abs(uva.coherency(X[iu[0]], X[iu[1]]))
    others = np.stack([np.delete(W, i, 0).mean(0) for i in range(n)])
    Cc = np.abs(uva.coherency(X, uva.mt_fft(others, nfft)))
    rng = np.random.default_rng(seed)
    null = []
    for _ in range(n_null):
        Ws = np.stack([np.roll(w, rng.integers(len(w))) for w in W])
        Xs = uva.mt_fft(Ws, nfft)
        null.append(np.abs(uva.coherency(X[iu[0]], Xs[iu[1]]))[band].max())
    return Ca[band].mean(), Cc[band].mean(), np.percentile(null, 95), f, Ca

In [ ]:
rend_rows = []
for bird in BIRDS:
    W5 = warped(bird, 5.0)
    W2 = warped(bird)
    a, b, c = corr_readings(W5)
    ca, cc_, null95, f, Ca = coherence_readings(W2, n_null=1000)
    rend_rows.append({"bird": bird, "renditions": len(W5), "corr_a_pairs": a, "corr_b_xcorr_peak": b,
                      "corr_c_vs_others": c, "coh_a_pairs_1_12": ca, "coh_c_vs_others_1_12": cc_,
                      "null95_max_1_12_pairs": null95})
rend_tab = pd.DataFrame(rend_rows)

In [ ]:
print(rend_tab.round(3).to_string(index=False))
print("\nMean over birds:")
print(rend_tab.drop(columns=["bird", "renditions"]).mean().round(3).to_string())
print("Paper: correlation 0.64 +- 0.06, coherence 0.68 +- 0.09 (1 to 12 Hz).")

**Result: partially reproduced (assumed channel).** Time-warped MUA is consistent across
renditions in all 3 birds, far above the shifted null (its 95th percentile of the 1 to 12 Hz
maximum is about 0.2 to 0.35). No reading puts both values in tolerance: all pairs give correlation 0.701
(paper 0.64 +- 0.06, just above) and coherence 0.585 (paper 0.68 +- 0.09, just below); against
the mean of the other renditions, 0.829 and 0.730.

In [ ]:
site_rows, site_pair_rows = [], []
for bird in BIRDS:
    R = renditions[bird]
    W5 = warped(bird, 5.0); W2 = warped(bird)
    sites = np.array([site_of(ev) for ev, _, _ in R["rows"]], dtype=object)
    days = np.array([ev["key"][1] for ev, _, _ in R["rows"]])
    names = [s for s in pd.unique(sites) if s is not None and np.sum(sites == s) >= 5]
    for s in names:
        site_rows.append({"bird": bird, "site": s, "renditions": int(np.sum(sites == s))})
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            si, sj = sites == names[i], sites == names[j]
            if days[si][0] != days[sj][0]:
                continue  # sites of one penetration: same day
            r_avg = np.corrcoef(W5[si].mean(0), W5[sj].mean(0))[0, 1]
            Zi = (W5[si] - W5[si].mean(1, keepdims=True)) / W5[si].std(1, keepdims=True)
            Zj = (W5[sj] - W5[sj].mean(1, keepdims=True)) / W5[sj].std(1, keepdims=True)
            r_single = (Zi @ Zj.T / W5.shape[1]).mean()
            nfft = 2 ** int(np.ceil(np.log2(W2.shape[1])) + 2)
            f = np.fft.rfftfreq(nfft, 1 / 1000)
            Xi, Xj = uva.mt_fft(W2[si], nfft), uva.mt_fft(W2[sj], nfft)
            ii, jj = np.meshgrid(np.arange(si.sum()), np.arange(sj.sum()), indexing="ij")
            ii, jj = ii.ravel(), jj.ravel()
            if len(ii) > 2000:
                k = np.random.default_rng(3).choice(len(ii), 2000, replace=False); ii, jj = ii[k], jj[k]
            coh = np.abs(uva.coherency(Xi[ii], Xj[jj]))
            band = (f >= 1) & (f <= 12)
            site_pair_rows.append({"bird": bird, "site_1": names[i], "site_2": names[j], "corr_a_site_averages": r_avg,
                                   "corr_b_single_renditions": r_single, "coh_b_single_renditions_1_12": coh[band].mean()})
print("Sites with at least 5 renditions:")
print(pd.DataFrame(site_rows).to_string(index=False))
sp = pd.DataFrame(site_pair_rows)
print("\nPairs of sites recorded on the same day:")
print(sp.round(3).to_string(index=False))
by_bird = sp.groupby("bird").mean(numeric_only=True)
print("\nPer bird (mean over pairs):\n", by_bird.round(3).to_string())
print("\nMean over birds with a pair:\n", by_bird.mean().round(3).to_string())
print("Paper: correlation 0.62 +- 0.03, coherence 0.65 +- 0.13. Coherence of two site averages with one taper is "
      "identically 1, so coherence is computed between single renditions of the two sites (reading b).")

**Result: partially reproduced (assumed channel).** Activity at sites of the same day is similar:
coherence between single renditions of two sites averages 0.688 over birds (paper 0.65 +- 0.13).
The correlation is higher than the paper's: 0.705 for single renditions and 0.893 for site
averages (paper 0.62 +- 0.03). For Uva4 the penetration of each site is not known.

### fig7a-correlation-with-amplitude, fig7b-10hz-power-peaks, fig7d-10hz-coherence

> Uva activity correlates with song amplitude (peak correlation 0.40) and leads it by 42 ms.

> Song amplitude and Uva activity both have a broad spectral peak near 10 Hz, at 8.0 Hz for song
> and 8.8 Hz for Uva.

> Uva activity and song amplitude are significantly coherent near 10 Hz (peak at 8.8 Hz,
> coherence 0.75, p < 0.01, phase -0.77 pi), with a large peak in their cross-spectrum.

Segments are (a) bouts of 1 to 8.192 s (the shortest bout is not given; 1 s, a standard choice,
so that the taper's 4 Hz resolution can show a 10 Hz peak) and (b) motif renditions (unwarped).
The amplitude readings are (a) dB, (b) power without the log, (c) rectified audio smoothed at
2 ms. Spectra are padded to 8192 points at 1 kHz; longer bouts are left out (counted).

In [ ]:
NFFT = 8192
FR = np.fft.rfftfreq(NFFT, 1 / 1000)
long_bouts = {}


def segments(bird, kind):
    out = []
    if kind == "bouts":
        for ev, m in bird_traces(bird):
            r = eph["recs"][ev["key"]]
            for a, b, _ in ev["bouts"]:
                a, b = int(a), int(b)
                if b - a > NFFT:
                    long_bouts[bird] = long_bouts.get(bird, 0) + 1
                    continue
                if b - a >= 1000 and b <= len(m):
                    out.append((m[a:b], r["amp_db"][a:b], 10 ** (r["amp_db"][a:b] / 10), r["amp_rect"][a:b]))
    else:
        for ev, m, pts in renditions[bird]["rows"]:
            r = eph["recs"][ev["key"]]
            a, b = int(pts[0]), int(pts[-1])
            out.append((m[a:b], r["amp_db"][a:b], 10 ** (r["amp_db"][a:b] / 10), r["amp_rect"][a:b]))
    return out


AMP = {"a": 1, "b": 2, "c": 3}
xc_rows = []
for bird in BIRDS:
    for kind in ["bouts", "motifs"]:
        segs = segments(bird, kind)
        for rd, j in AMP.items():
            lags = np.arange(-150, 151)
            cc = []
            for s in segs:
                x, y = s[0].astype(float), s[j].astype(float)
                vals = []
                for L in lags:  # positive lag: MUA earlier than amplitude
                    if L >= 0:
                        xx, yy = x[:len(x) - L], y[L:]
                    else:
                        xx, yy = x[-L:], y[:len(y) + L]
                    vals.append(np.corrcoef(xx, yy)[0, 1])
                cc.append(vals)
            cc = np.nanmean(cc, 0)
            k = int(np.argmax(np.abs(cc)))
            xc_rows.append({"bird": bird, "segments": kind, "amplitude": rd, "n": len(segs),
                            "peak_corr": cc[k], "lag_ms_mua_first": lags[k], "corr_at_0": cc[150]})
xc = pd.DataFrame(xc_rows)

In [ ]:
print(xc.round(3).to_string(index=False))
print("Bouts longer than 8.192 s left out (counted once per reading):", long_bouts)
print("\nMean over birds (peak magnitude, lag):")
print(xc.assign(mag=xc.peak_corr.abs()).groupby(["segments", "amplitude"])[["mag", "lag_ms_mua_first"]].mean().round(3).to_string())
print("Paper: 0.40 +- 0.04 at 42 +- 6 ms, MUA first.")

**Result: reproduced (assumed channel).** With motif segments and the rectified amplitude
(reading c), the peak correlation averages 0.382 over birds with MUA leading by 40.7 ms (paper
0.40 +- 0.04 at 42 +- 6). With bouts the same amplitude gives 0.459 at 40.7 ms; the dB amplitude
gives larger correlations (0.55 to 0.60).

In [ ]:
spec_rows, coh_store = [], {}
for bird in BIRDS:
    for kind in ["bouts", "motifs"]:
        segs = segments(bird, kind)
        for rd, j in AMP.items():
            Xm = np.concatenate([uva.mt_fft(s[0].astype(float), NFFT) for s in segs])
            Xa = np.concatenate([uva.mt_fft(s[j].astype(float), NFFT) for s in segs])
            lens = np.array([len(s[0]) for s in segs], float)
            Pm = np.abs(Xm[:, 0]) ** 2; Pa = np.abs(Xa[:, 0]) ** 2
            coh_store[(bird, kind, rd)] = (Xm, Xa)
            for norm in ["sum", "length"]:
                if norm == "sum":
                    pm = (Pm / Pm.sum(1, keepdims=True)).mean(0); pa = (Pa / Pa.sum(1, keepdims=True)).mean(0)
                else:
                    pm = (Pm / lens[:, None]).mean(0); pa = (Pa / lens[:, None]).mean(0)
                spec_rows.append({"bird": bird, "segments": kind, "amplitude": rd, "norm": norm, "pm": pm, "pa": pa})
spec = pd.DataFrame(spec_rows)

In [ ]:
win = (FR > 3) & (FR < 30)
rows = []
for (kind, rd, norm), g in spec.groupby(["segments", "amplitude", "norm"]):
    # each bird's spectrum as the reading normalizes it, averaged over birds without renormalizing
    pm, pa = np.mean(list(g.pm), 0), np.mean(list(g.pa), 0)
    per_bird_m = [FR[win][np.argmax(p[win])] for p in g.pm]
    per_bird_a = [FR[win][np.argmax(p[win])] for p in g.pa]
    rows.append({"segments": kind, "amplitude": rd, "norm": norm, "peak_song_Hz": FR[win][np.argmax(pa[win])],
                 "peak_mua_Hz": FR[win][np.argmax(pm[win])],
                 "per_bird_song": np.round(per_bird_a, 1).tolist(), "per_bird_mua": np.round(per_bird_m, 1).tolist()})
print(pd.DataFrame(rows).to_string(index=False))
print("Paper: song 8.0 Hz, MUA 8.8 Hz. Search window 3 to 30 Hz.")

**Result: partially reproduced (assumed channel).** With the spectra normalized by their sum,
over motifs both peak near 9 Hz: MUA at 8.9 Hz (paper 8.8) and song amplitude at 9.0 Hz (dB) or
9.3 Hz (power or rectified), above the song tolerance (7.2 to 8.8 Hz). Normalized by length and
averaged without renormalizing, the MUA peak moves to 12.6 Hz over motifs and 11.6 Hz over bouts,
and over bouts the song amplitude peaks at 6.1 to 6.2 Hz. No reading puts both peaks within
tolerance.


In [ ]:
rng = np.random.default_rng(7)
rows = []
for kind in ["bouts", "motifs"]:
    for rd in AMP:
        cohs, nulls = [], []
        for bird in BIRDS:
            Xm, Xa = coh_store[(bird, kind, rd)]
            cohs.append(uva.coherency(Xm, Xa))
        C = np.mean(cohs, 0)
        # null: MUA of each segment shifted circularly by a random amount, 1000 times
        segs_by_bird = {b: [s[0].astype(float) for s in segments(b, kind)] for b in BIRDS}
        nmax = []
        for it in range(1000):
            cn = []
            for bird in BIRDS:
                Xa = coh_store[(bird, kind, rd)][1]
                Xs = uva.shifted_ffts(segs_by_bird[bird], NFFT, rng)
                cn.append(uva.coherency(Xs, Xa))
            nmax.append(np.abs(np.mean(cn, 0))[win].max())
        k = np.argmax(np.abs(C)[win])
        rows.append({"segments": kind, "amplitude": rd, "peak_Hz": FR[win][k], "coherence": np.abs(C)[win][k],
                     "phase_pi": np.angle(C[win][k]) / np.pi, "null_99_max": np.percentile(nmax, 99),
                     "p": (np.sum(np.array(nmax) >= np.abs(C)[win][k]) + 1) / (len(nmax) + 1), "null_draws": len(nmax),
                     "per_bird_peak_Hz": [round(FR[win][np.argmax(np.abs(c)[win])], 1) for c in cohs]})
coh_tab = pd.DataFrame(rows)
print(coh_tab.round(3).to_string(index=False))
print("Paper: 8.8 Hz, coherence 0.75, p < 0.01, phase -0.77 pi. Phase as Chronux coherencyc(MUA, amplitude) "
      "returns it; the opposite convention (amplitude first) is the same value with the sign changed.")

**Result: reproduced (assumed channel).** Over motifs with the amplitude as power (reading b),
the averaged coherency peaks at 9.0 Hz with coherence 0.789 and phase -0.721 pi, above the null
of shifted MUA (p = 0.001, 1000 shifts; paper 8.8 Hz, 0.75, -0.77 pi, p < 0.01). With the dB or
rectified amplitude the peak is at 8.8 to 8.9 Hz but the coherence is higher (0.85). Over bouts
the peak is at 6.3 Hz.

### spectral-features-weak

> None of six spectral song features correlates significantly with Uva activity within a bird;
> pooled over birds only amplitude (r = 0.18) and gravity center (r = 0.17) do, after Bonferroni
> correction.

The six features are computed from the sound channel at 1 kHz after the Sound Analysis Pro
definitions (`uva.sap_features`; the claims file adopts SAP as an assumption, and the details of
the window, taper, band, AM and FM are standard choices stated in the helper). Windows run from
10 ms after each selected song syllable's onset to 10 ms before its offset, and the MUA window
is the same window 42 ms earlier. Reading (a): per rendition, mean feature against mean MUA;
reading (b): per 1 ms bin within the windows. Pearson r, Bonferroni over 6 features (p < 0.05/6).
Pooled: all birds' renditions together, raw and (variant, reading a) with features and MUA
z-scored within each bird's syllable type.


In [ ]:
rows, pooled = [], {"a": [], "a_z": [], "b": []}
for bird in BIRDS:
    per_a, per_b = [], []
    for ev, m in bird_traces(bird):
        F = eph["recs"][ev["key"]]["features"]
        for i in np.where(syl_mask(ev))[0]:
            a, z = int(ev["on"][i]) + 10, int(ev["off"][i]) - 10
            if z - a < 5 or a - 42 < 0 or z > len(m):
                continue
            mu = m[a - 42:z - 42]
            per_a.append([ev["title"][i], mu.mean()] + [F[f][a:z].mean() for f in uva.FEATURES])
            per_b.append(np.c_[mu, np.stack([F[f][a:z] for f in uva.FEATURES], 1)])
    A = pd.DataFrame(per_a, columns=["type", "mua"] + uva.FEATURES)
    B = np.concatenate(per_b)
    Az = A.copy()
    for col in ["mua"] + uva.FEATURES:
        Az[col] = A.groupby("type")[col].transform(lambda v: (v - v.mean()) / v.std() if v.std() > 0 else v * 0)
    pooled["a"].append(A); pooled["a_z"].append(Az.dropna()); pooled["b"].append(B)
    for j, f in enumerate(uva.FEATURES):
        ra = stats.pearsonr(A.mua, A[f]); rb = stats.pearsonr(B[:, 0], B[:, j + 1])
        rows.append({"bird": bird, "feature": f, "renditions": len(A), "r_a": ra.statistic, "p_a": ra.pvalue,
                     "sig_a": ra.pvalue < 0.05 / 6, "bins": len(B), "r_b": rb.statistic, "p_b": rb.pvalue,
                     "sig_b": rb.pvalue < 0.05 / 6})
sf = pd.DataFrame(rows)
print("Per bird:")
print(sf.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
prow = []
for rd in ["a", "a_z", "b"]:
    if rd == "b":
        X = np.concatenate(pooled["b"]); mua_, feats_ = X[:, 0], {f: X[:, j + 1] for j, f in enumerate(uva.FEATURES)}
    else:
        X = pd.concat(pooled[rd]); mua_, feats_ = X.mua.values, {f: X[f].values for f in uva.FEATURES}
    for f in uva.FEATURES:
        r = stats.pearsonr(mua_, feats_[f])
        prow.append({"reading": rd, "feature": f, "n": len(mua_), "r": r.statistic, "p": r.pvalue,
                     "significant_bonferroni": r.pvalue < 0.05 / 6})
print("\nPooled over birds:")
print(pd.DataFrame(prow).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nFeatures significant in single birds, per reading:",
      {rd: int(sf[f"sig_{rd}"].sum()) for rd in ["a", "b"]}, "of", len(sf), "bird-feature pairs")
print("Paper: none significant in single birds; pooled, amplitude r = 0.18 and gravity center r = 0.17 significant, "
      "the other four not.")


**Result: not reproduced (assumed feature definitions).** Within single birds, 9 of 18
bird-feature pairs are significant per rendition (reading a) and 15 of 18 per 1 ms bin (reading
b); the paper reports none. Pooled over birds, amplitude correlates with MUA (r = 0.156 per
rendition, 0.183 per bin, the latter within 0.18 +- 0.018), but so do the other features: gravity
center r = 0.291 (a) and 0.082 (b), outside 0.17 +- 0.017, and pitch goodness, Wiener entropy and
FM are significant under readings (a) and (b). With each feature z-scored within syllable type,
pitch goodness and FM stay significant and Wiener entropy does not (p = 0.06). The features
follow SAP's definitions only as far as the helper states. The attempt record keeps this claim
blocked, the outcome round 2 of verification reviewed, and proposes not reproduced for a third
round.


### fig9-long-syllable-gamma

> In long syllables (> 150 ms; 13 syllables, 43% of song syllables) Uva activity can oscillate
> fast; 8 of 13 have a significant spectral peak above 10 Hz, at 17 to 51 Hz, and Uva activity is
> coherent with song amplitude from 1 to 55 Hz.

Syllable types (labels) of each bird whose median duration exceeds 150 ms. For each type, MUA
and amplitude (reading a, dB) from onset to 50 ms before offset, spectra with [2 1] tapers padded
to 1024 points, normalized by their sum and averaged over renditions. Null for the MUA spectrum:
the 1 kHz MUA samples of each window scrambled and smoothed again with the 2 ms Gaussian, taking
the scrambled window as periodic so that the smoothing adds no dip at its edges (the paper
scrambles the raw signal; the reduced trace is scrambled here), 1000 times; a type has a
significant peak when its largest local maximum above 10 Hz exceeds the 95th percentile of the
null's maxima above 10 Hz. Coherence with amplitude is averaged over renditions per type, then
over all long types, against the null of scrambled MUA averaged the same way (95th percentile
of its maxima over 1 to 60 Hz, 200 draws).

In [ ]:
NF9 = 1024
F9 = np.fft.rfftfreq(NF9, 1 / 1000)
hi = F9 > 10
band9 = (F9 >= 1) & (F9 <= 60)
rng = np.random.default_rng(9)
long_rows, coh9, null9 = [], [], []
n_types = 0
for bird in BIRDS:
    durs = {}
    for ev, m in bird_traces(bird):
        for t, a, b, s in zip(ev["title"], ev["on"], ev["off"], ev["sel"]):
            if s and t not in ("i", "k", ""):
                durs.setdefault(t, []).append((ev, m, a, b))
    for t, items in sorted(durs.items()):
        if len(items) < 5:
            continue
        n_types += 1
        med = np.median([b - a for _, _, a, b in items])
        if med <= 150:
            continue
        segs = [(m[int(a):int(b) - 50].astype(float), eph["recs"][ev["key"]]["amp_db"][int(a):int(b) - 50].astype(float))
                for ev, m, a, b in items if b - a - 50 >= 50 and int(b) - 50 <= len(m)]
        Xm = np.concatenate([uva.mt_fft(x, NF9) for x, _ in segs])
        Xa = np.concatenate([uva.mt_fft(y, NF9) for _, y in segs])
        P = np.abs(Xm[:, 0]) ** 2
        P = (P / P.sum(1, keepdims=True)).mean(0)
        nullmax = []
        nc = []
        for it in range(1000):
            Xs = uva.scrambled_ffts([x for x, _ in segs], NF9, rng)
            Ps = np.abs(Xs[:, 0]) ** 2
            Ps = (Ps / Ps.sum(1, keepdims=True)).mean(0)
            nullmax.append(Ps[hi].max())
            if it < 200:
                nc.append(np.abs(uva.coherency(Xs, Xa)))
        thr = np.percentile(nullmax, 95)
        # the largest local maximum above 10 Hz (a peak, not the edge of the band)
        locmax = np.r_[False, (P[1:-1] > P[:-2]) & (P[1:-1] > P[2:]), False] & hi
        k = np.argmax(np.where(locmax, P, -np.inf)[hi])
        C = np.abs(uva.coherency(Xm, Xa))
        coh9.append(C); null9.append(np.array(nc))
        thr_t = np.percentile(np.array(nc)[:, band9].max(1), 95)
        ok_t = C > thr_t
        span = (F9 >= 1.1) & (F9 <= 49.5)
        long_rows.append({"bird": bird, "type": t, "median_ms": med, "renditions": len(segs),
                          "peak_above_10Hz": F9[hi][k], "peak_power": P[hi][k], "null95_max": thr,
                          "significant": bool(P[hi][k] > thr), "coh_null95": thr_t,
                          "coh_above_null_1.1_49.5Hz": f"{ok_t[span].mean():.2f}", "coh_unbroken_1.1_49.5Hz": bool(ok_t[span].all())})
lt = pd.DataFrame(long_rows)
print(lt.round(5).to_string(index=False))
print(f"\nLong types: {len(lt)} of {n_types} syllable types with at least 5 renditions ({len(lt) / n_types:.2f}); "
      f"paper 13 (43%) over 5 birds")
sig = lt[lt.significant]
print(f"Long types with a significant peak above 10 Hz: {len(sig)} of {len(lt)} ({len(sig) / len(lt):.2f}); "
      f"peak frequencies {sorted(sig.peak_above_10Hz.round(1).tolist())}; paper 8 of 13 (0.62), 17 to 51 Hz")
Cm = np.mean(coh9, 0)
thr_c = np.percentile(np.mean(null9, 0)[:, band9].max(1), 95)  # null of the across-type average, maxima over 1-60 Hz
above = Cm > thr_c
band = (F9 >= 1) & (F9 <= 60)
runs = []
start = None
for fv, ab in zip(F9[band], above[band]):
    if ab and start is None:
        start = fv
    if not ab and start is not None:
        runs.append((start, prev)); start = None
    prev = fv
if start is not None:
    runs.append((start, prev))
print(f"Average coherence over long types above the null band (95th percentile of the averaged null's maxima, "
      f"{thr_c:.3f}), ranges in Hz within 1 to 60 Hz: {[(round(a, 1), round(b, 1)) for a, b in runs]}")
print("Coherence at 1, 5, 10, 20, 30, 40, 50, 55 Hz:", [round(float(np.interp(v, F9, Cm)), 3) for v in [1, 5, 10, 20, 30, 40, 50, 55]])
# The criterion's across-bird average: long types averaged within each bird, then over birds (the
# null averaged the same way, draw by draw).
birds9 = list(dict.fromkeys(lt.bird))
Cb = np.mean([np.mean([c for c, b in zip(coh9, lt.bird) if b == bb], 0) for bb in birds9], 0)
Nb = np.mean([np.mean([n for n, b in zip(null9, lt.bird) if b == bb], 0) for bb in birds9], 0)
thr_b = np.percentile(Nb[:, band9].max(1), 95)
runs_b, start = [], None
for fv, ab in zip(F9[band9], (Cb > thr_b)[band9]):
    if ab and start is None:
        start = fv
    if not ab and start is not None:
        runs_b.append((start, prev)); start = None
    prev = fv
if start is not None:
    runs_b.append((start, prev))
print(f"Across-bird average ({len(birds9)} birds) above its null ({thr_b:.3f}), ranges in Hz within 1 to 60 Hz: "
      f"{[(round(float(a), 1), round(float(b), 1)) for a, b in runs_b]}; unbroken over 1.1 to 49.5 Hz: "
      f"{bool((Cb > thr_b)[span].all())}")
print("Across-bird coherence at 1, 5, 10, 20, 30, 40, 50, 55 Hz:", [round(float(np.interp(v, F9, Cb)), 3) for v in [1, 5, 10, 20, 30, 40, 50, 55]])


**Result: not reproduced (assumed channel).** Over 3 birds, 6 of 18 syllable types are
longer than 150 ms (0.33; paper 0.43 +- 0.043). 5 of the 6 have a significant MUA peak above 10 Hz
(0.83; paper 0.62 +- 0.062), at 10.7 to 40.0 Hz (lower bound outside 15.3 to 56.1 Hz). With the
scrambled null smoothed without edge effects, the coherence with amplitude averaged over the
long types exceeds its null (0.107) from 2.0 to 59.6 Hz without a break, covering 1.1 to 49.5 Hz;
averaged within each bird and then over the 3 birds, as the criterion says, it exceeds its null
(0.141) over the same range. The coherence part holds; the counts and the frequency range do not.


### bout-offset-suppression

> Uva activity is transiently suppressed for about 200 ms after bout offsets.

MUA aligned to bout offsets (the end of the last segment of each bout holding song syllables),
against the shuffled baseline drawn at random times outside singing (more than 300 ms from any
segment) in the same files, with the 95% band of the shuffles' minima over -100 to +500 ms.

In [ ]:
rows = []
for bird in BIRDS:
    rng = np.random.default_rng(21)
    A, pool = [], []
    for ev, m in bird_traces(bird):
        song = syl_mask(ev)
        offs = np.array([b for a, b, idx in ev["bouts"] if song[idx].any()])
        A.append(uva.aligned(m, offs, 100, 500)[0])
        alls = np.zeros(len(m), bool)
        for a, b in zip(ev["on"], ev["off"]):
            alls[max(0, int(a) - 300):int(b) + 300] = True
        t = np.where(~alls)[0]
        t = t[(t >= 100) & (t < len(m) - 501)]
        if len(t):
            pool.append((m, t))
    A = np.concatenate(A)
    avg = A.mean(0)
    lags = np.arange(-100, 501)
    level, lo, hi_ = uva.shuffle_band(pool, len(A), 100, 500, 1000, rng)
    AVG[("bout offsets", bird, "a")] = (avg, level, lo, hi_)
    below = avg < lo
    post = lags >= 0
    first = np.argmax(below & post) if np.any(below & post) else None
    if first is not None:
        j = first
        while j + 1 < len(avg) and avg[j + 1] < lo:
            j += 1
        end_below = lags[j]
        back = np.where((lags > lags[first]) & (avg >= lo))[0]
        dur_in = lags[back[0]] if len(back) else np.nan
        k = np.where((lags > lags[first]) & (avg >= level))[0]
        dur_level = lags[k[0]] if len(k) else np.nan
    else:
        end_below, dur_in, dur_level = np.nan, np.nan, np.nan
    rows.append({"bird": bird, "bout_offsets": len(A), "baseline_level": level, "band_lo": lo, "min_after": avg[post].min(),
                 "min_at_ms": lags[post][np.argmin(avg[post])], "first_below_ms": lags[first] if first is not None else np.nan,
                 "back_in_band_ms": dur_in, "back_to_level_ms": dur_level})
bo = pd.DataFrame(rows)
print(bo.round(4).to_string(index=False))
print(f"Mean over birds: returns into the band {bo.back_in_band_ms.mean():.0f} ms after offset; "
      f"returns to the baseline level {bo.back_to_level_ms.mean():.0f} ms (paper about 200 ms, tolerance 180 to 220)")

**Result: partially reproduced (assumed channel).** MUA falls below the non-singing band after
bout offsets in all 3 birds, but returns into the band 30, 309 and 42 ms after offset (mean 127
ms; paper about 200, tolerance 180 to 220). The suppression is present; its duration is outside
the tolerance and varies widely between birds.


### Event-aligned averages

Each bird's average MUA around calls, introductory notes, syllable onsets and offsets, and bout
offsets, with the 95% bands of the shuffled baselines that the claims above use.


In [ ]:
import matplotlib.pyplot as plt

names = ["calls", "introductory notes", "syllable onsets", "syllable offsets", "bout offsets"]
fig, axs = plt.subplots(len(BIRDS), len(names), figsize=(18, 9), sharex="col")
for j, nm in enumerate(names):
    for i, bird in enumerate(BIRDS):
        ax = axs[i, j]
        for rd, col in [("a", "C0"), ("b", "C1")]:
            if (nm, bird, rd) not in AVG:
                continue
            avg, level, lo, hi_ = AVG[(nm, bird, rd)]
            lags = np.arange(-100, 501) if nm == "bout offsets" else LAGS
            if rd == "a":
                ax.plot(lags, avg, "k", lw=1)
                print(f"{nm}, {bird}: average {avg.min():.4f} to {avg.max():.4f} V; band (shuffle a) {lo:.4f} to {hi_:.4f}")
            ax.axhspan(lo, hi_, color=col, alpha=0.15)
            ax.axhline(level, color=col, lw=0.6)
        ax.axvline(0, color="grey", lw=0.5)
        if i == 0:
            ax.set_title(nm)
        if j == 0:
            ax.set_ylabel(f"{bird}\nMUA (V)")
for ax in axs[-1]:
    ax.set_xlabel("ms from event")
plt.tight_layout()
plt.show()
print("Bands: blue shuffle (a), orange shuffle (b); lines are the shuffles' mean level. Every average is within its axes.")


### no-motif-bout-signal

> Uva shows little activation related to the onsets or offsets of motifs or bouts.

Per rendition, mean MUA in the 40 ms before onset (the window of the fig6 test) for motif onsets
(first syllable of a motif rendition), bout onsets (the first song syllable of a bout, after any
introductory notes) and other syllable onsets. One-sided two-sample t-test (motif or bout
larger) at p < 0.05, the claims file's standard choice. After motif and bout offsets, the
offset-aligned average is compared with the upper band of the shuffled baseline (inside bouts)
from 0 to +150 ms.

In [ ]:
rows = []
for bird in BIRDS:
    R = renditions[bird]
    motif_on = {}
    motif_off = {}
    for ev, m, pts in R["rows"]:
        motif_on.setdefault(ev["key"], []).append(pts[0]); motif_off.setdefault(ev["key"], []).append(pts[-1])
    v_motif, v_bout, v_other = [], [], []
    off_motif, off_bout = [], []
    for ev, m in bird_traces(bird):
        song = syl_mask(ev)
        bout_first = set()
        for a, b, idx in ev["bouts"]:
            s = [i for i in idx if song[i]]
            if s:
                bout_first.add(s[0])
                off_bout.append(ev["off"][s[-1]])
        mo = set(np.round(motif_on.get(ev["key"], [])).astype(int))
        for i in np.where(song)[0]:
            t = int(round(ev["on"][i]))
            if t < 40:
                continue
            v = m[t - 40:t].mean()
            if t in mo:
                v_motif.append(v)
            elif i in bout_first:
                v_bout.append(v)
            else:
                v_other.append(v)
        off_motif += [(m, t) for t in motif_off.get(ev["key"], [])]
    tm = stats.ttest_ind(v_motif, v_other, alternative="greater")
    tb = stats.ttest_ind(v_bout, v_other, alternative="greater")
    rng = np.random.default_rng(31)
    exc = {}
    for name, items in [("motif offsets", off_motif)]:
        A = np.concatenate([uva.aligned(m, [t], PRE, POST)[0] for m, t in items])
        level, lo, hi_ = uva.shuffle_band(inside_bout_pool(bird), len(A), PRE, POST, 1000, rng)
        exc[name] = float((A.mean(0)[PRE:PRE + 151] - hi_).max())
    Ab = []
    for ev, m in bird_traces(bird):
        song = syl_mask(ev)
        offs = [ev["off"][[i for i in idx if song[i]][-1]] for a, b, idx in ev["bouts"] if any(song[i] for i in idx)]
        Ab.append(uva.aligned(m, offs, PRE, POST)[0])
    Ab = np.concatenate(Ab)
    level, lo, hi_ = uva.shuffle_band(inside_bout_pool(bird), len(Ab), PRE, POST, 1000, rng)
    exc["bout offsets"] = float((Ab.mean(0)[PRE:PRE + 151] - hi_).max())
    rows.append({"bird": bird, "n_motif_on": len(v_motif), "n_bout_on": len(v_bout), "n_other": len(v_other),
                 "motif_on": np.mean(v_motif), "bout_on": np.mean(v_bout), "other_on": np.mean(v_other),
                 "p_motif_gt_other": tm.pvalue, "p_bout_gt_other": tb.pvalue,
                 "max_excess_after_motif_off": exc["motif offsets"], "max_excess_after_bout_off": exc["bout offsets"]})
nm = pd.DataFrame(rows)
print(nm.to_string(index=False))
print("A positive excess means the offset-aligned average rises above the band's upper edge after the offset.")

**Result: not reproduced (assumed channel).** MUA before motif onsets is significantly higher than
before other syllable onsets in Orange150 (0.0635 against 0.0607, p = 0.0012), though not in b871
(p = 0.46) or Uva4 (p = 0.18). Bout onsets are not higher in any bird (p 0.78 to 0.99), and no
average rises above the band after motif or bout offsets (largest excess below zero in all).

## Lesion data

The lesion archive holds, for each of 7 birds, a folder of song before the lesion and one after
it, each with the authors' analysis file (segment times at 44.1 kHz, and which segments are
selected) and the .wav files it lists. The archive is streamed twice: first for the analysis
files, then for the audio, of which only the amplitude traces and the bouts chosen for the
maturity index are kept.

In [ ]:
t0 = time.time()
les = uva.load_lesion_dbases()
print(f"Lesion analysis files read in {time.time() - t0:.0f} s")


def syllables(db):
    """Selected segments per file (ms), sorted, and the file name."""
    out = []
    for f in db["files"]:
        s = f["seg"][f["selected"]] / db["fs"] * 1000.0
        if len(s):
            out.append((f["sound_file"], s[np.argsort(s[:, 0])]))
    return out


LKEYS = sorted(les["dbase"])
LBIRDS = sorted({k[0] for k in LKEYS})
summary = []
for k in LKEYS:
    db = les["dbase"][k]
    T = np.array([f["time"] for f in db["files"]])
    syl = syllables(db)
    durs = np.concatenate([s[:, 1] - s[:, 0] for _, s in syl])
    days = pd.Series(pd.to_datetime(T - 719529, unit="D")).dt.date.value_counts().to_dict()
    summary.append({"bird": k[0], "condition": k[1], "fs": db["fs"], "files": len(db["files"]),
                    "segments": sum(len(f["seg"]) for f in db["files"]), "selected": len(durs),
                    "dates": {str(d): n for d, n in sorted(days.items())},
                    "span_h": (T.max() - T.min()) * 24, "audio_min": sum(f["length"] for f in db["files"]) / db["fs"] / 60,
                    "median_syl_ms": np.median(durs)})
summ = pd.DataFrame(summary)

In [ ]:
print(summ.round(2).to_string(index=False))
print("\nOther files:")
for name, v in les["other"].items():
    print("  ", name, v["bytes"], "bytes, MD5", les["md5"][name])
for k, v in les["syllgaps"].items():
    db = les["dbase"][k]
    syl = syllables(db)
    durs = np.concatenate([s[:, 1] - s[:, 0] for _, s in syl])
    print(f"\nSyllGaps.mat in {k}: {len(v['durs'])} durations (median {np.median(v['durs']):.1f} ms) and {len(v['gaps'])} gaps; "
          f"the analysis file's selected segments: {len(durs)} (median {np.median(durs):.1f} ms). The two differ; "
          "SyllGaps.mat is not described by the data descriptor and is shown as a variant only.")

In [ ]:
SEED_MI = 1
rng_mi = np.random.default_rng(SEED_MI)
chosen, keep = {}, {}
for k in LKEYS:
    db = les["dbase"][k]
    fs = db["fs"]
    for gap in (300, 350):
        cands = []
        for f in db["files"]:
            s = f["seg"][f["selected"]]  # samples
            for b in uva.bouts_from_segments(s, gap / 1000 * fs):
                a, z = s[b, 0].min(), s[b, 1].max()
                if 0.7 * fs <= z - a <= 2 * fs:
                    cands.append((f["sound_file"], int(a), int(z)))
        pick = [cands[i] for i in sorted(rng_mi.choice(len(cands), min(100, len(cands)), replace=False))]
        chosen[(k, gap)] = (len(cands), pick)
        for w, a, z in pick:
            keep.setdefault((k[0], k[1], w), set()).add((a, z))
keep = {k: sorted(v) for k, v in keep.items()}
t0 = time.time()
aud = uva.load_lesion_audio(keep)
print(f"Lesion audio streamed in {time.time() - t0:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

In [ ]:
print("Sampling rates of the .wav files:", pd.Series(list(aud["fs"].values())).value_counts().to_dict(),
      "; analysis files: 44100")
listed = {(k[0], k[1], f["sound_file"]) for k in LKEYS for f in les["dbase"][k]["files"]}
have = set(aud["amp"])
print(f"Files listed in the analysis files: {len(listed)}; .wav files in the archive: {len(have)}; "
      f"listed but missing: {len(listed - have)}; present but not listed: {len(have - listed)}")
lens_ok = [abs(len(aud["amp"][(k[0], k[1], f["sound_file"])][0]) - f["length"] / 44.1) <= 2
           for k in LKEYS for f in les["dbase"][k]["files"] if (k[0], k[1], f["sound_file"]) in have]
print(f"Files whose length matches the analysis file's FileLength: {sum(lens_ok)} of {len(lens_ok)}")
wav_md5 = pd.Series(aud["md5"])
print("wav files with identical content to another:", int(wav_md5.duplicated(keep=False).sum()))
# Clock check: amplitude (dB) inside selected segments against outside, true and shifted labels.
rows = []
for k in LKEYS:
    db = les["dbase"][k]
    r = {"bird": k[0], "condition": k[1]}
    for shift in [0, 200, 1000]:
        ins, outs = [], []
        for w, s in syllables(db)[:300]:
            a = aud["amp"][(k[0], k[1], w)][0]
            mask = np.zeros(len(a), bool)
            for x, y in s + shift:
                mask[int(max(0, x)):int(min(len(a), y))] = True
            if mask.any() and (~mask).any():
                ins.append(a[mask].mean()); outs.append(a[~mask].mean())
        r[f"dB in minus out, shift {shift} ms"] = np.mean(ins) - np.mean(outs)
    rows.append(r)
print(pd.DataFrame(rows).round(2).to_string(index=False))

The analysis files hold one recording day before and one after the lesion for each bird (two
post-lesion folders span two calendar dates, recordings that run past midnight). So the claims'
readings of which post-lesion day to use (first day, or all days pooled) are the same here. The
data do not say whether this post-lesion day is the first day of singing after surgery.

In [ ]:
# Is directed song marked anywhere? File names, their parts, recording hours and properties.
for k in LKEYS:
    db = les["dbase"][k]
    names = [f["sound_file"] for f in db["files"]]
    T = np.array([f["time"] for f in db["files"]])
    hours = pd.Series(pd.to_datetime(T - 719529, unit="D")).dt.hour
    fields = {len(n[:-4].split("_")) for n in names}
    print(f"{k[0]} {k[1]}: {len(names)} files, e.g. {names[0]} ... {names[-1]}; fields per name {sorted(fields)}; "
          f"properties {db['property_names'] or 'none'}; files per hour of day {hours.value_counts().sort_index().to_dict()}")
print("Names are <bird>_<MATLAB date number>_<month>_<day>_<hour>_<minute>_<second>.wav; no part of them, and no "
      "property, marks a recording as directed or undirected.")


### lesion-undirected-rate-similar

> In social isolation, Uva-lesioned birds sing at rates similar to intact birds.

Recording hours are the span from the first to the last file of the day (the recordings are
triggered by sound, so silent hours are not recorded; the span is the closest measure of time
available). Rates per hour of span: syllables, bouts (gaps up to 300 ms) and singing time (bout
durations), counted on the selected segments (the authors' song syllables) and, as a variant, on
all segments. Rates per hour of recorded audio (summed file lengths) are shown too.

In [ ]:
rows = []
for k in LKEYS:
    db = les["dbase"][k]
    T = np.array([f["time"] for f in db["files"]])
    span = (T.max() - T.min()) * 24
    syl = syllables(db)
    n_syl = sum(len(s) for _, s in syl)
    bouts = [(s[b, 0].min(), s[b, 1].max()) for _, s in syl for b in uva.bouts_from_segments(s, 300)]
    # all segments (selected or not), the authors' segmentation before their selection
    alls = [f["seg"][np.argsort(f["seg"][:, 0])] / db["fs"] * 1000.0 for f in db["files"] if len(f["seg"])]
    n_all = sum(len(s) for s in alls)
    bouts_all = [(s[b, 0].min(), s[b, 1].max()) for s in alls for b in uva.bouts_from_segments(s, 300)]
    audio_h = sum(f["length"] for f in db["files"]) / db["fs"] / 3600
    rows.append({"bird": k[0], "condition": k[1], "span_h": span, "audio_h": audio_h,
                 "selected_fraction": n_syl / n_all,
                 "syllables_per_h": n_syl / span, "bouts_per_h": len(bouts) / span,
                 "singing_s_per_h": sum(b - a for a, b in bouts) / 1000 / span,
                 "all_segments_per_h": n_all / span, "all_bouts_per_h": len(bouts_all) / span,
                 "all_singing_s_per_h": sum(b - a for a, b in bouts_all) / 1000 / span,
                 "all_segments_per_audio_h": n_all / audio_h, "syllables_per_audio_h": n_syl / audio_h,
                 "syllables": n_syl, "bouts": len(bouts), "all_segments": n_all, "all_bouts": len(bouts_all)})
rate = pd.DataFrame(rows)
print(rate.round(2).to_string(index=False))
for col in ["syllables_per_h", "bouts_per_h", "singing_s_per_h", "all_segments_per_h", "all_bouts_per_h",
            "all_singing_s_per_h", "syllables_per_audio_h", "all_segments_per_audio_h", "syllables", "bouts",
            "all_segments", "all_bouts"]:
    p = rate.pivot(index="bird", columns="condition", values=col)
    w = stats.wilcoxon(p["post-lesion"], p["pre-lesion"])
    print(f"{col}: post/pre ratio per bird {np.round((p['post-lesion'] / p['pre-lesion']).values, 2).tolist()}; "
          f"Wilcoxon signed-rank p = {w.pvalue:.3f}")

**Result: not reproduced (assumed measure).** Counting the selected segments per hour of the
recording span, lesioned birds sing less after the lesion: syllables per hour fall in 6 of 7
birds (post/pre ratio 0.08 to 1.01; Wilcoxon p = 0.031), as do bouts per hour (p = 0.047) and
singing time (p = 0.031). The result depends on the count: with all segments, selected or not,
the decreases are not significant (p = 0.078, 0.109 and 0.078), and the selected fraction differs
between days (Purple53 0.59 before, 0.34 after). Per hour of recorded audio, all segments give p =
1.000 and selected segments p = 0.078, but recordings are triggered by sound, so recorded audio
is not a measure of time. Spans range from 2.2 to 41.6 h.


### fig2g-maturity-index-drop

> Uva lesions lower the maturity index, a measure of song repeatability, from 0.26 to 0.08
> (Wilcoxon signed-rank test, p < 0.025, 7 birds).

100 bouts of 0.7 to 2 s drawn at random (seed fixed) per bird and condition, for bout gaps of
300 and 350 ms. Spectrograms by the Methods' parameters, 860 Hz to 8.6 kHz. The claims file's
readings (a) and (b) of the lag correlation function are run. The paper says "correlation of
power spectra"; read literally (Pearson correlation of each pair of slices' power spectra), the
silent and noisy slices of every bout correlate through the shared shape of the background
spectrum. Whether the spectra are log-scaled and whether the mean spectrum is removed before
correlating is not stated, and these change the outcome, so four forms are run: power or log
power, each as is or with the mean over all slices of the bird's condition removed from each
frequency (and divided by its SD). See the deviations for the reading adopted. The Methods
cite Aronov et al. 2008 for this measure; its supplement could not be read in this run (HTTP 403).

In [ ]:
mi_rows = []
t0 = time.time()
for k in LKEYS:
    for gap in (300, 350):
        n_c, pick = chosen[(k, gap)]
        specs = [uva.spectrogram_mt(aud["raw"][((k[0], k[1], w), a, z)].astype(float), 44100) for w, a, z in pick]
        for form in ["power", "log", "power centred", "log centred"]:
            S = [np.log(s + 1e-12) for s in specs] if form.startswith("log") else specs
            if form.endswith("centred"):
                allS = np.concatenate(S)
                mu, sd = allS.mean(0), allS.std(0)
                S = [(s - mu) / sd for s in S]
            a_, b_ = uva.maturity_index(S)
            mi_rows.append({"bird": k[0], "condition": k[1], "gap": gap, "form": form, "candidates": n_c,
                            "bouts": len(pick), "MI_a": a_, "MI_b": b_})
mi = pd.DataFrame(mi_rows)
print(f"(computed in {time.time() - t0:.0f} s)")
res = []
for (gap, form), g in mi.groupby(["gap", "form"]):
    for rd in ["MI_a", "MI_b"]:
        p = g.pivot(index="bird", columns="condition", values=rd)
        w = stats.wilcoxon(p["post-lesion"], p["pre-lesion"])
        res.append({"gap": gap, "form": form, "lag reading": rd[-1], "pre_mean": p["pre-lesion"].mean(),
                    "pre_sd": p["pre-lesion"].std(), "post_mean": p["post-lesion"].mean(), "post_sd": p["post-lesion"].std(),
                    "birds_decreasing": int((p["post-lesion"] < p["pre-lesion"]).sum()), "wilcoxon_p": w.pvalue,
                    "pre_in_tol": abs(p["pre-lesion"].mean() - 0.26) <= 0.03, "post_in_tol": abs(p["post-lesion"].mean() - 0.08) <= 0.02})
mi_res = pd.DataFrame(res)
print(mi_res.round(3).to_string(index=False))
print("\nPer bird, log centred, gap 300 ms, reading a:")
print(mi[(mi.gap == 300) & (mi.form == "log centred")].pivot(index="bird", columns="condition", values="MI_a").round(3).to_string())
print("\nPer bird, power (literal reading), gap 300 ms, reading a:")
print(mi[(mi.gap == 300) & (mi.form == "power")].pivot(index="bird", columns="condition", values="MI_a").round(3).to_string())
print("Paper: pre 0.26 +- 0.03, post 0.08 +- 0.02, p < 0.025, 7 birds.")

**Result: partially reproduced, resting on an assumption.** With log spectra centred on the
condition's mean spectrum, lag reading (a) and 300 ms gaps, MI falls from 0.235 to 0.096 (paper
0.26 +- 0.03 and 0.08 +- 0.02), in all 7 birds (Wilcoxon p = 0.016). That form is this attempt's,
not a reading the claims file lists, and it is the only one of the 16 combinations that puts
both values in tolerance (with 350 ms gaps the pre value is 0.213, just outside). Read literally
(raw power or log power, not centred), MI does not fall (2 or fewer birds decrease) and its
values (0.32 to 0.62) are far from the paper's. The Methods take the measure from Aronov et al.
2008, whose supplement could not be read in this run, so the decrease rests on the adopted form.


### fig2f-closer-to-exponential

> After Uva lesions syllable-duration distributions are much closer to exponential (goodness of
> fit Gamma 3.7 against 16 before; Wilcoxon signed-rank test, p < 0.025, 7 birds), but stay above
> the subsong criterion of Gamma < 2.

Syllables are the selected segments of each day. Gamma readings (a) D times the square root of
n and (b) D over the Lilliefors critical value at alpha 0.05 (1.06 / sqrt(n) for the
exponential, Lilliefors 1969). Ref 41 (Aronov et al. 2008) could not be read in this run (its
supplement returned HTTP 403), so its definition could not be checked. Variant: the durations of SyllGaps.mat (one bird).

In [ ]:
rows = []
for k in LKEYS:
    durs = np.concatenate([s[:, 1] - s[:, 0] for _, s in syllables(les["dbase"][k])])
    g = uva.gamma_readings(durs)
    rows.append({"bird": k[0], "condition": k[1], "n": g["n"], "D": g["D"], "Gamma_a": g["a"], "Gamma_b": g["b"]})
gam = pd.DataFrame(rows)
print(gam.round(3).to_string(index=False))
for rd in ["Gamma_a", "Gamma_b", "D"]:
    p = gam.pivot(index="bird", columns="condition", values=rd)
    w = stats.wilcoxon(p["post-lesion"], p["pre-lesion"])
    print(f"{rd}: pre {p['pre-lesion'].mean():.2f} +- {p['pre-lesion'].std():.2f}, post {p['post-lesion'].mean():.2f} +- "
          f"{p['post-lesion'].std():.2f}; birds decreasing {(p['post-lesion'] < p['pre-lesion']).sum()} of 7; Wilcoxon p = {w.pvalue:.4f}")
for k, v in les["syllgaps"].items():
    g = uva.gamma_readings(v["durs"])
    print(f"Variant, SyllGaps.mat {k}: n = {g['n']}, D = {g['D']:.3f}, Gamma_a = {g['a']:.2f}, Gamma_b = {g['b']:.2f}")
print("Paper: pre 16 +- 6, post 3.7 +- 1.2, p < 0.025; post above 2.")

**Result: partially reproduced.** Syllable durations are closer to exponential after the lesion in
all 7 birds (Wilcoxon p = 0.016 under both readings) and stay above 2, but Gamma is about twice
the paper's: reading (a) 30.4 before and 15.2 after (paper 16 +- 6 and 3.7 +- 1.2), reading (b)
28.7 and 14.3.

### fig2c-small-syllable-peak

> In 6 of 7 Uva-lesioned birds the post-lesion syllable-duration distribution has a small peak
> at 50 to 100 ms.

In [ ]:
rows = []
for k in [k for k in LKEYS if k[1] == "post-lesion"]:
    durs = np.concatenate([s[:, 1] - s[:, 0] for _, s in syllables(les["dbase"][k])])
    for bw in (5, 10):
        edges = np.arange(0, 1000 + bw, bw)
        h, _ = np.histogram(durs, edges)
        c = edges[:-1] + bw / 2
        expd = len(durs) * (np.exp(-edges[:-1] / durs.mean()) - np.exp(-edges[1:] / durs.mean()))
        w = (c >= 50) & (c <= 100)
        loc = np.r_[False, (h[1:-1] > h[:-2]) & (h[1:-1] >= h[2:]), False]
        a_ = bool(np.any(loc & w & (h > expd)))
        hi95 = stats.poisson.ppf(0.975, expd)
        b_ = bool(np.any(loc & w & (h > hi95)))
        rows.append({"bird": k[0], "bin_ms": bw, "n": len(durs), "local_max_50_100_ms": c[loc & w].tolist(),
                     "reading_a": a_, "reading_b": b_})
pk = pd.DataFrame(rows)
print(pk.to_string(index=False))
for bw in (5, 10):
    s = pk[pk.bin_ms == bw]
    print(f"bins {bw} ms: reading (a) {s.reading_a.sum()} of 7, reading (b) {s.reading_b.sum()} of 7 (paper 6 of 7)")

In [ ]:
fig, axs = plt.subplots(1, 7, figsize=(21, 3), sharex=True)
for ax, k in zip(axs, [k for k in LKEYS if k[1] == "post-lesion"]):
    durs = np.concatenate([s[:, 1] - s[:, 0] for _, s in syllables(les["dbase"][k])])
    edges = np.arange(0, 1005, 5)
    h, _ = np.histogram(durs, edges)
    expd = len(durs) * (np.exp(-edges[:-1] / durs.mean()) - np.exp(-edges[1:] / durs.mean()))
    c = edges[:-1] + 2.5
    ax.bar(c, h, width=5, color="0.6")
    ax.plot(c, expd, "C0", lw=1, label="fitted exponential")
    ax.plot(c, stats.poisson.ppf(0.975, expd), "C0--", lw=1, label="its 97.5% bound")
    ax.axvspan(50, 100, color="C1", alpha=0.15)
    ax.set_xlim(0, 300); ax.set_title(k[0], fontsize=9); ax.set_xlabel("syllable duration (ms)")
    print(f"{k[0]}: {int((durs > 300).sum())} of {len(durs)} syllables beyond 300 ms (off the axes); "
          f"largest bin count {h.max()}")
axs[0].set_ylabel("syllables per 5 ms bin"); axs[0].legend(fontsize=7)
plt.tight_layout(); plt.show()


**Result: partially reproduced.** With 5 ms bins all 7 birds have a local maximum at 50 to 100 ms
above the fitted exponential and above its 97.5% Poisson bound (both readings); with 10 ms bins
(variant) 6 of 7 do, as the paper reports. The count under the main bins is 7, one more than the
paper's 6. The histograms show that several of these maxima are bumps of one or two bins (LPink15
has four between 50 and 100 ms), so the 5 ms count is sensitive to count noise.


### fig2d-gaps-broader

> After Uva lesions gap durations become broadly distributed, with more long and more short
> gaps, like subsong.

In [ ]:
rows = []
for b in LBIRDS:
    g = {}
    for cond in ["pre-lesion", "post-lesion"]:
        gs = []
        for _, s in syllables(les["dbase"][(b, cond)]):
            d = s[1:, 0] - s[:-1, 1]
            gs.append(d[d <= 300])
        g[cond] = np.concatenate(gs)
    lo, hi5 = np.percentile(g["pre-lesion"], [5, 95])
    post = g["post-lesion"]
    iqr = lambda x: np.subtract(*np.percentile(np.log(x[x > 0]), [75, 25]))
    rows.append({"bird": b, "gaps_pre": len(g["pre-lesion"]), "gaps_post": len(post), "pre_p5_ms": lo, "pre_p95_ms": hi5,
                 "post_frac_short": np.mean(post < lo), "post_frac_long": np.mean(post > hi5),
                 "post_frac_outside": np.mean((post < lo) | (post > hi5)),
                 "iqr_log_pre": iqr(g["pre-lesion"]), "iqr_log_post": iqr(post)})
gp = pd.DataFrame(rows)
print(gp.round(3).to_string(index=False))
print(f"Reading (a): birds with more than 10% outside {(gp.post_frac_outside > 0.10).sum()} of 7; both short and long fractions "
      f"above 5%: {((gp.post_frac_short > 0.05) & (gp.post_frac_long > 0.05)).sum()} of 7")
w = stats.wilcoxon(gp.post_frac_outside - 0.10)
print(f"   Wilcoxon on (fraction outside - 0.10): p = {w.pvalue:.4f}")
w2 = stats.wilcoxon(gp.iqr_log_post, gp.iqr_log_pre)
print(f"Reading (b): IQR of log gap larger after in {(gp.iqr_log_post > gp.iqr_log_pre).sum()} of 7; Wilcoxon p = {w2.pvalue:.4f}")

**Result: reproduced.** After the lesion 16 to 38% of gaps fall outside the bird's pre-lesion 5th
to 95th percentiles, against 10% if unchanged (Wilcoxon p = 0.016); the long fraction rises in
all 7 birds and the short fraction exceeds 5% in 6 of 7. The interquartile range of log gap
widens in 5 of 7 birds (p = 0.078).

### fig2e-rhythmicity-drop and fig2e-residual-rhythm-peak

> Uva lesions decrease song rhythmicity, R, from 3.30 to 1.7 (Wilcoxon signed-rank test,
> p < 0.025, 7 birds).

> Unlike subsong, 6 of 7 Uva-lesioned birds keep a significant peak in the song rhythm spectrum
> between 3 and 8 Hz.

Bouts with gaps of at most 350 ms and at least 1 s long (the shortest bout is not given; a
standard choice). Amplitude readings (a) dB and (c) the log of the rectified, 2 ms smoothed audio
(the Song rhythmicity section log-scales the amplitude, so reading (b), without the log, is not
run for R). Null readings (a) the condition's own syllable and gap distributions, (b) the
post-lesion distributions; synthetic bouts match each real bout's length; levels are the mean
amplitude inside and outside syllables. R is the maximum of the ratio between 3 and 30 Hz (the
upper bound is not given; a standard choice). The peak test draws 1000 synthetic bout sets, as
the claims file asks. For post-lesion song the two null readings are the same, so the peak test
is run once. A maximum of the ratio at the band's edge is not a peak, so the test is also run
requiring a local maximum inside the band, and, as a variant, on the peak's prominence.

In [ ]:
def rhythm_inputs(k, minlen=1000):
    db = les["dbase"][k]
    B, durs, gaps = [], [], []
    for w, s in syllables(db):
        durs += list(s[:, 1] - s[:, 0])
        d = s[1:, 0] - s[:-1, 1]
        gaps += list(d[d <= 350])
        dbt, rect = aud["amp"][(k[0], k[1], w)]
        for b in uva.bouts_from_segments(s, 350):
            a, z = int(s[b, 0].min()), int(s[b, 1].max())
            if z - a >= minlen and z <= len(dbt) and z - a <= uva.NFFT_RHYTHM:
                inside = np.zeros(z - a, bool)
                for x, y in s[b] - a:
                    inside[int(x):int(y)] = True
                B.append({"a": dbt[a:z].astype(float), "c": np.log(rect[a:z].astype(float) + 1e-9), "inside": inside})
    gaps = np.array(gaps)
    return B, np.array(durs), gaps[gaps > 0]


FRH = np.fft.rfftfreq(uva.NFFT_RHYTHM, 1 / 1000)
N_PEAK_DRAWS = 1000  # the claims file's number of null draws
peak_store = {}
rh_in = {k: rhythm_inputs(k) for k in LKEYS}
rh_rows, peak_rows = [], []
for k in LKEYS:
    B, durs, gaps = rh_in[k]
    for amp in ["a", "c"]:
        spec = np.mean([uva.bout_spectrum(b[amp]) for b in B], 0)
        lev_s = np.mean([b[amp][b["inside"]].mean() for b in B])
        lev_g = np.mean([b[amp][~b["inside"]].mean() for b in B if (~b["inside"]).any()])
        for null in ["a", "b"]:
            src = k if null == "a" else (k[0], "post-lesion")
            d_, g_ = rh_in[src][1], rh_in[src][2]
            rng = np.random.default_rng([LKEYS.index(k), ord(amp), ord(null)])
            lg = np.log(g_)

            def draw():
                return np.mean([uva.bout_spectrum(uva.synthetic_bout(len(b[amp]), d_.mean(), lg.mean(), lg.std(),
                                                                     lev_s, lev_g, rng)) for b in B], 0)
            nullspec = np.mean([draw() for _ in range(5)], 0)
            ratio = spec / nullspec
            m = (FRH > 3) & (FRH <= 30)
            rh_rows.append({"bird": k[0], "condition": k[1], "amplitude": amp, "null": null, "bouts": len(B),
                            "R": ratio[m].max(), "R_freq": FRH[m][np.argmax(ratio[m])],
                            "R_3_50": ratio[(FRH > 3) & (FRH <= 50)].max()})
            if k[1] == "post-lesion" and amp == "a" and null == "a":
                rr = []
                for _ in range(N_PEAK_DRAWS):
                    rr.append(draw() / nullspec)
                rr = np.array(rr)
                peak_store[k[0]] = (ratio, rr)
                for lo_, hi_ in [(3, 8), (4, 8)]:
                    bm = (FRH >= lo_) & (FRH <= hi_)
                    thr_max = np.percentile(rr[:, bm].max(1), 95)
                    thr_f = np.percentile(rr[:, bm], 95, axis=0)
                    peak_rows.append({"bird": k[0], "null": null, "band": f"{lo_}-{hi_} Hz", "max_ratio": ratio[bm].max(),
                                      "at_Hz": FRH[bm][np.argmax(ratio[bm])], "null95_of_max": thr_max,
                                      "significant_max": bool(ratio[bm].max() > thr_max),
                                      "significant_any_freq": bool(np.any(ratio[bm] > thr_f))})
rh = pd.DataFrame(rh_rows)

In [ ]:
print(rh.round(3).to_string(index=False))
for (amp, null), g in rh.groupby(["amplitude", "null"]):
    p = g.pivot(index="bird", columns="condition", values="R")
    w = stats.wilcoxon(p["post-lesion"], p["pre-lesion"])
    print(f"amplitude {amp}, null {null}: R pre {p['pre-lesion'].mean():.2f} +- {p['pre-lesion'].std():.2f}, post "
          f"{p['post-lesion'].mean():.2f} +- {p['post-lesion'].std():.2f}; decreasing in {(p['post-lesion'] < p['pre-lesion']).sum()} "
          f"of 7; Wilcoxon p = {w.pvalue:.4f}")
print("Paper: pre 3.30 +- 1.1, post 1.7 +- 0.3, p < 0.025.")

**Result: partially reproduced.** With the dB amplitude, R falls in all 7 birds (Wilcoxon p =
0.016 under both nulls), but the values are higher than the paper's: 4.84 before and 2.44 after
with null (a) (paper 3.30 +- 1.1 and 1.7 +- 0.3), and 5.87 and 2.41 with null (b). The rectified
amplitude gives unstable ratios dominated by frequencies near 30 Hz and no consistent decrease.

In [ ]:
def band_peak(r, bm):
    """Largest local maximum of the ratio strictly inside the band (higher than both neighbours,
    so not the band's edge), and its prominence within the band: its height above the higher
    of the lowest points on either side of it in the band. (nan, 0) when there is none."""
    idx = np.where(bm)[0]
    best, prom = np.nan, 0.0
    for j in idx[1:-1]:
        if r[j] > r[j - 1] and r[j] > r[j + 1] and not (r[j] <= best):
            best = r[j]
            prom = r[j] - max(r[idx[0]:j + 1].min(), r[j:idx[-1] + 1].min())
    return best, prom


pkr = pd.DataFrame(peak_rows)
print("Maximum of the ratio anywhere in the band (the test as first run; a maximum at the band's edge counts):")
print(pkr.round(3).to_string(index=False))
for (null, band_), g in pkr.groupby(["null", "band"]):
    print(f"null {null}, {band_}: significant (max over band) {g.significant_max.sum()} of 7; "
          f"(any frequency, per-frequency 95th percentile) {g.significant_any_freq.sum()} of 7 (paper 6 of 7)")
rows = []
for bird, (ratio, rr) in peak_store.items():
    for lo_, hi_ in [(3, 8), (4, 8)]:
        bm = (FRH >= lo_) & (FRH <= hi_)
        pk_, pr_ = band_peak(ratio, bm)
        null_pk = np.array([band_peak(r, bm) for r in rr])
        null_max = rr[:, bm].max(1)
        thr_pk = np.percentile(null_max, 95)  # a local maximum must exceed the null's maximum anywhere in the band
        thr_pr = np.percentile(null_pk[:, 1], 95)
        at = FRH[np.where(bm & (ratio == pk_))[0][0]] if np.isfinite(pk_) else np.nan
        rows.append({"bird": bird, "band": f"{lo_}-{hi_} Hz", "local_max": pk_, "at_Hz": at, "null95_of_max": thr_pk,
                     "a_local_max_significant": bool(pk_ > thr_pk), "prominence": pr_, "null95_prominence": thr_pr,
                     "null_draws_with_a_local_max": int(np.isfinite(null_pk[:, 0]).sum()),
                     "b_prominence_significant": bool(pr_ > thr_pr)})
lp = pd.DataFrame(rows)
print(f"\nWith a local maximum required ({N_PEAK_DRAWS} null draws per bird):")
print(lp.round(3).to_string(index=False))
for band_, g in lp.groupby("band"):
    print(f"{band_}: local maximum above the null's 95th percentile of maxima {g.a_local_max_significant.sum()} of 7; "
          f"prominence above the null's 95th percentile of prominences {g.b_prominence_significant.sum()} of 7 (paper 6 of 7)")

fig, axs = plt.subplots(1, 7, figsize=(21, 3), sharey=True)
for ax, (bird, (ratio, rr)) in zip(axs, peak_store.items()):
    m = (FRH > 1) & (FRH <= 30)
    ax.plot(FRH[m], ratio[m], "k", lw=1)
    ax.fill_between(FRH[m], np.percentile(rr[:, m], 2.5, axis=0), np.percentile(rr[:, m], 97.5, axis=0), color="C0", alpha=0.3)
    ax.axvspan(3, 8, color="C1", alpha=0.1)
    ax.set_title(bird, fontsize=9); ax.set_xlabel("Hz")
    print(f"{bird}: ratio 1 to 30 Hz ranges {ratio[m].min():.2f} to {ratio[m].max():.2f}")
axs[0].set_ylabel("post-lesion spectrum / null"); axs[0].set_ylim(0, 4)
plt.tight_layout(); plt.show()
print("Black: real ratio; blue: 2.5 to 97.5 percentiles of the null draws' ratio per frequency; shaded 3 to 8 Hz.")


**Result: not reproduced.** No version of the test gives the paper's 6 of 7. Taking the maximum
of the ratio anywhere in the band, all 7 birds pass, but in 3 of them (Purple94, LPink13, LPink15)
that maximum is the band's lower edge, 3.17 Hz. Requiring a local maximum inside the band, all 7
still pass (local maxima 1.29 to 3.34 against null thresholds of about 1.1 to 1.3, 1000 draws): the
synthetic null has no amplitude texture within syllables, so the whole real ratio lies above it.
Requiring the peak's prominence to exceed the null's prominences, 4 of 7 pass (5 of 7 between 4
and 8 Hz).


## Blocked claims

- **lesion-directed-song-abolished**: the paper and the data descriptor say directed song was
  recorded, but neither the file names (bird, date number, date and time; printed in the lesion
  data check), the analysis files (no properties) nor the descriptor mark any recording as
  directed. If directed recordings are in the folders, they cannot be told apart. Blocked,
  metadata insufficient.
- **figs1-control-lesions-no-effect**: the archive holds 7 lesioned birds, the paper's Uva lesion
  count; there are no control-lesion birds. Blocked, data missing.
- **single-unit-spontaneous-rates**: the analysis files hold no spike times (EventTimes empty) and
  no sorted units; the spreadsheets mark 32 "spontaneous activity" recordings, which are raw
  multiunit signals with no unit identities and nothing to say which units project to HVC.
  Blocked, metadata insufficient.
- **fig3-antidromic**: the stimulation files are described in the cell below. Stimulus times can
  be taken from the artefacts, which come back to within 0.25 V of baseline a median 2.3 ms after
  their onset (1.3 to 8.7 ms over 40 files), inside the claim's 1.5 to 5 ms window; 36 of 779
  stimuli have recovered before 1.5 ms. The claim's
  latency and jitter are those of identified single units, measured from stimulus times. The files
  hold no stimulus times and no sorted or identified units, and in a multiunit trace a response
  locked to the stimulus with under 100 us of jitter repeats on every trial like the artefact, so
  it cannot be told apart from it without the units and collision tests. The claim is not tested.
  Blocked, other. An earlier version of the cell counted spikes after subtracting each file's
  median trace; that removes any response locked to the stimulus, so the count was dropped.
- **fig8-no-gte-relation**: needs GTE times from the model-based method of Boari et al. (ref 50).
  The paper links no code. A third-party copy of GTE extraction code from the Mindlin lab exists
  (github.com/kylerbrown/mindlin_auto); it is not the paper's implementation, was not tried here,
  and whether it runs is not known. Blocked, other (not attempted).


In [ ]:
# fig3-antidromic: the stimulation artefacts and how long they last against the 1.5 to 5 ms window
rows = []
for _, r in sheet[sheet.note.str.contains("stim")].iterrows():
    key = (r.bird, r.day, int(r.file))
    if key not in eph["recs"] or CHAN[r.bird] not in eph["recs"][key]["stim"]:
        continue
    sn, on = eph["recs"][key]["stim"][CHAN[r.bird]]
    t = (np.arange(sn.shape[1]) - 200) / 40.0  # ms from the artefact's first saturated sample
    base = np.median(sn[:, :180], axis=1, keepdims=True)
    dev = np.abs(sn - base) > 0.25  # V; the baseline's spread is about 0.05 V
    rec = np.array([t[np.where(d & (t <= 10))[0][-1]] if d.any() else np.nan for d in dev])
    w = (t >= 1.5) & (t <= 5)
    rows.append({"bird": r.bird, "day": r.day, "file": int(r.file), "note": r.note, "stimuli": len(sn),
                 "recovery_ms_median": np.nanmedian(rec), "recovered_before_1.5ms": int((rec < 1.5).sum()),
                 "frac_1.5_5ms_beyond_0.25V": dev[:, w].mean()})
art = pd.DataFrame(rows)
print(art.round(2).to_string(index=False))
print(f"\nStimulation files with artefacts on the chosen channel: {len(art)}; median recovery to within 0.25 V of "
      f"baseline {art.recovery_ms_median.median():.1f} ms (range {art.recovery_ms_median.min():.1f} to "
      f"{art.recovery_ms_median.max():.1f}); fraction of the 1.5 to 5 ms window beyond 0.25 V, median "
      f"{art['frac_1.5_5ms_beyond_0.25V'].median():.2f}")
print(f"Stimuli whose artefact is back within 0.25 V before 1.5 ms: {art['recovered_before_1.5ms'].sum()} of {art.stimuli.sum()}")
print("Intensities in the spreadsheet notes:", sorted(set(art.note.str.extract(r"(\d+)\s*uA")[0].dropna())))
fig, axs = plt.subplots(1, 4, figsize=(16, 3), sharey=True)
for ax, (_, r) in zip(axs, art.iloc[np.linspace(0, len(art) - 1, 4).astype(int)].iterrows()):
    sn, _ = eph["recs"][(r.bird, r.day, r.file)]["stim"][CHAN[r.bird]]
    t = (np.arange(sn.shape[1]) - 200) / 40.0
    ax.plot(t, sn.T, lw=0.4); ax.axvspan(1.5, 5, color="C1", alpha=0.15)
    ax.set_xlim(-1, 10); ax.set_ylim(-1, 1); ax.set_title(f"{r.bird} {r.day} file {r.file}", fontsize=9)
    ax.set_xlabel("ms from artefact onset")
axs[0].set_ylabel("V (clipped to +-1)")
plt.tight_layout(); plt.show()
print("The traces saturate at +-10 V; the axes show +-1 V, so the saturated part lies outside them.")


## Summary

| Claim | Outcome |
|---|---|
| lesion-directed-song-abolished | blocked (metadata insufficient) |
| lesion-undirected-rate-similar | not reproduced (assumption) |
| fig2g-maturity-index-drop | partially reproduced (assumption) |
| fig2f-closer-to-exponential | partially reproduced |
| fig2c-small-syllable-peak | partially reproduced |
| fig2d-gaps-broader | reproduced |
| fig2e-rhythmicity-drop | partially reproduced |
| fig2e-residual-rhythm-peak | not reproduced |
| figs1-control-lesions-no-effect | blocked (data missing) |
| fig3-antidromic | blocked (other: no stimulus times or identified units) |
| single-unit-spontaneous-rates | blocked (metadata insufficient) |
| fig4c-call-premotor | partially reproduced (assumption) |
| fig4b-introductory-note-premotor | partially reproduced (assumption) |
| fig4a-elevated-during-song | reproduced (assumption) |
| fig5-consistent-across-renditions | partially reproduced (assumption) |
| fig5-consistent-across-sites | partially reproduced (assumption) |
| fig6-peak-before-onset | partially reproduced (assumption) |
| fig6-dip-before-offset | reproduced (assumption) |
| fig7a-correlation-with-amplitude | reproduced (assumption) |
| spectral-features-weak | blocked in the record (verified); not reproduced here (assumption), proposed for round 3 |
| fig7b-10hz-power-peaks | partially reproduced (assumption) |
| fig7d-10hz-coherence | reproduced (assumption) |
| fig8-no-gte-relation | blocked (other: not attempted) |
| fig9-long-syllable-gamma | not reproduced (assumption) |
| bout-offset-suppression | partially reproduced (assumption) |
| no-motif-bout-signal | not reproduced (assumption) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The electrophysiology covers 3 of the paper's 5
birds. Each recording has three neural channels and nothing in the files settles which is the
Uva electrode; the channel with the largest syllable-onset modulation is used for each bird (an
assumption that every electrophysiology outcome rests on; for Orange150 it is the channel the
authors' analysis file ran its event detectors on). MUA is computed from the raw signal as the
Methods say, then sampled at 1 kHz. Several methods the paper leaves open were run under each
reading the claims file lists; the maturity index needed a form the claims file does not list
(log spectra centred on the mean spectrum), adopted as an assumption because the literal reading
cannot give values near the paper's. The song features follow Sound Analysis Pro's definitions
as far as the helper states. The rhythm null and the shuffled baselines are built as the claims
file describes; the synthetic rhythm null has no amplitude texture within syllables.

**What was found about the data set.** The lesion data match their analysis files exactly (12,833
wav files, all listed, all lengths equal). The electrophysiology archive holds a second copy of
one recording day as a nested archive (173 identical files). The descriptor says introductory
notes and calls are unselected, but they are selected and labelled "i" and "k"; it does not say
which of the three neural channels is the recording electrode, nor what the labels mean. No
lesion recording is marked as directed or undirected. One Uva4 recording (file 20) has an
unreadable closing sample count. Two vector clustering files that the descriptor does not
mention have the same size in two birds' folders. The spreadsheet tab for Uva4 is named "October
6, 2016" and lists dates from 2007-07-25 to 2007-07-27, while the analysis file runs to
2007-08-06.
